# AIC 2026 — Pipeline Online (bản CHẠY NỘP BÀI — không debug/thử nghiệm)
Notebook này chỉ gồm: setup môi trường -> định nghĩa các hàm xử lý (Query Understanding,
Visual/Text/Object/Audio Search, RRF Fusion, Diversify, Multimodal Rerank, Temporal
Reasoning, xuất CSV) -> 1 cell chạy để xử lý gói câu hỏi BTC cấp và tự động tải file `.zip`
nộp bài. Không còn cell debug, hiển thị ảnh kiểm tra, hay chạy thử với câu hỏi mẫu.

**Cách dùng:** Runtime -> Restart and run all (chạy từ đầu tới cuối). Trước khi chạy tới
Phần 15, upload các file `.txt` BTC cấp vào thư mục `/content/query_package/` (tên file
phải có hậu tố đúng quy ước: `...-kis.txt`, `...-qa.txt`, `...-trake.txt`).

Notebook này thực hiện luồng xử lý ONLINE: nhận 1 câu query text, tra cứu trên **Index Store** đã build sẵn từ notebook Offline, tự động nhận diện `task_type` và trả về đúng định dạng nộp bài tương ứng:
- **Textual KIS**: `<video_id, frame_id>` xếp hạng, tối đa 100 dòng.
- **QA**: `<video_id, frame_id, answer>` — answer sinh bằng Gemini Vision (LVLM Reasoning) cho top-5.
- **TRAKE**: `<video_id, frame_id_1...N>` — Dense Frame Access + Gemini Vision căn chỉnh từng event.

**Nguyên tắc quan trọng:** CLIP model dùng để encode QUERY ở đây phải **CÙNG MODEL** với CLIP đã dùng để encode ảnh ở Offline (`clip-ViT-B-32`) — nếu khác model, 2 loại vector sẽ không nằm chung không gian số, so sánh vô nghĩa.

**LLM dùng xuyên suốt:** Google Gemini (`gemini-3.5-flash-lite`) — cho cả Query Understanding, Multimodal Reranker, LVLM Reasoning (QA), và Sequence matching (TRAKE). SDK dùng là `google-genai` (SDK mới, thay cho `google-generativeai` đã bị khai tử).

---

## Phần 0 — Setup: Mount Drive, cài thư viện

**Giải thích:**
- `sentence-transformers` — cung cấp model `clip-ViT-B-32`, đúng model BTC dùng để tạo CLIP features (Text Encoder + Image Encoder cùng 1 không gian vector).
- `underthesea` — thư viện xử lý tiếng Việt (tách từ, gán nhãn từ loại) — dùng cho Query Understanding.
- `faiss-cpu`, `rank_bm25` — đọc lại 2 loại index đã build ở Offline.

In [70]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [71]:
!pip install faiss-cpu rank_bm25 pyarrow sentence-transformers underthesea --quiet

In [72]:
import os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import faiss

DATASET_ROOT = "/content/drive/MyDrive/AIC 2026/dataset"
EXTRACTED_ROOT = os.path.join(DATASET_ROOT, "extracted")

print("EXTRACTED_ROOT:", EXTRACTED_ROOT)

EXTRACTED_ROOT: /content/drive/MyDrive/AIC 2026/dataset/extracted


---
## Phần 1 — Load Index Store (kết quả từ notebook Offline)

**Giải thích:** đây chính là bước "tải sổ mục lục đã chuẩn bị sẵn" — mọi thứ nặng (CLIP encode, Object Detection, OCR, ASR) đã chạy xong ở Offline, ở đây chỉ cần **load vào RAM 1 lần**, dùng lại cho mọi query sau đó.

In [ ]:
import pandas as pd
import os, pickle, faiss

# 1. FAISS Index (CLIP vectors)
clip_index = faiss.read_index(os.path.join(EXTRACTED_ROOT, "clip_index", "clip_faiss.index"))
clip_mapping_df = pd.read_parquet(os.path.join(EXTRACTED_ROOT, "clip_index", "clip_mapping.parquet"))
print(f"CLIP FAISS Index: {clip_index.ntotal} vector")

# 2. Objects Table
objects_df = pd.read_parquet(os.path.join(EXTRACTED_ROOT, "objects", "objects_index.parquet"))

# 3. Load Media Info (Từ file người dùng cung cấp)
# media_full.csv chứa title, description, author, length...
media_info_df = pd.read_csv('/content/drive/MyDrive/AIC 2026/dataset/media_info_analysis/media_full.csv')
print(f"Media Info: {len(media_info_df)} video")

# 4. Bảng map-keyframes (đã bỏ bước load 2 file worklist — không dùng ở đâu trong pipeline,
# và Media_Info_Exploitation.ipynb không còn tạo ra 2 file đó nữa)
map_keyframes_df = pd.read_parquet(os.path.join(EXTRACTED_ROOT, "map_keyframes", "map_keyframes_index.parquet"))

# Xử lý ASR/OCR fallback
bm25_path = os.path.join(EXTRACTED_ROOT, "final_index", "bm25.pkl")
text_index_path = os.path.join(EXTRACTED_ROOT, "final_index", "text_index.parquet")
if os.path.exists(bm25_path):
    with open(bm25_path, "rb") as f: bm25_index = pickle.load(f)
    text_index_df = pd.read_parquet(text_index_path)
else:
    bm25_index = None
    text_index_df = pd.DataFrame(columns=["video_id", "frame_index", "text", "source"])

---
## Phần 2 — Load CLIP model (dùng để encode QUERY)

**Giải thích:** đây chính là **CLIP Text Encoder** đã nhắc nhiều lần — nhận câu chữ, biến thành vector 512 chiều, CÙNG không gian số với các vector ảnh đã có sẵn trong FAISS Index.

In [74]:
from sentence_transformers import SentenceTransformer

clip_model = SentenceTransformer('clip-ViT-B-32')
print("Đã load CLIP model (clip-ViT-B-32) — dùng để encode cả text lẫn ảnh")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Đã load CLIP model (clip-ViT-B-32) — dùng để encode cả text lẫn ảnh


---
## Phần 3 — Query Understanding (dùng Gemini + underthesea)

**Giải thích:** kiến trúc 2 bước cố định, KHÔNG phải agent:
1. **Rule-based** — `underthesea.word_tokenize()` tách từ tiếng Việt trước (giải quyết vấn đề "dấu cách không phải ranh giới từ").
2. **1 lần gọi LLM duy nhất** (Gemini) — đưa câu đã tách từ + prompt kèm JSON schema, LLM trả về TOÀN BỘ phần còn lại trong 1 lần: `task_type, N, entities, actions, attributes, scene, temporal_relations, question, expanded_queries`.

**Dùng SDK mới `google-genai`** (SDK cũ `google-generativeai` đã bị Google khai tử, không dùng nữa).

**Cần chuẩn bị trước khi chạy:**
1. Lấy API key tại [Google AI Studio](https://aistudio.google.com/apikey).
2. Trên Colab: bấm biểu tượng 🔑 (Secrets) ở thanh bên trái → thêm secret tên `GEMINI_API_KEY`, dán API key vào → bật toggle "Notebook access".
3. Không hardcode API key trực tiếp vào code (tránh lộ khi chia sẻ notebook).

In [75]:
!pip install google-genai --quiet

In [76]:
from google.colab import userdata
from google import genai
from google.genai import types

# 2 API key riêng biệt (đã lưu trong Colab Secrets — KHÔNG hardcode)
GEMINI_API_KEY_1 = userdata.get('GEMINI_API_KEY1')   # dùng cho LLM 1: trích xuất, Reranker, QA, TRAKE
GEMINI_API_KEY_2 = userdata.get('GEMINI_API_KEY2')   # dùng RIÊNG cho LLM 2: giám khảo

gemini_client_1 = genai.Client(api_key=GEMINI_API_KEY_1)
gemini_client_2 = genai.Client(api_key=GEMINI_API_KEY_2)

GEMINI_MODEL = "gemini-3.5-flash-lite"       # LLM 1 — trích xuất, ưu tiên nhanh/rẻ
GEMINI_MODEL_JUDGE = "gemini-3.5-flash"       # LLM 2 — giám khảo, cần suy luận kỹ hơn 1 chút
# Nếu 2 model này bị deprecated, kiểm tra tên model hiện tại tại:
# https://ai.google.dev/gemini-api/docs/models

print("Đã khởi tạo 2 Gemini client (2 key riêng biệt)")
print("  LLM 1 (trích xuất/Reranker/QA/TRAKE):", GEMINI_MODEL)
print("  LLM 2 (giám khảo)                    :", GEMINI_MODEL_JUDGE)

Đã khởi tạo 2 Gemini client (2 key riêng biệt)
  LLM 1 (trích xuất/Reranker/QA/TRAKE): gemini-3.5-flash-lite
  LLM 2 (giám khảo)                    : gemini-3.5-flash


---
## Phần 2.1 — Rate limiter + retry dùng CHUNG cho mọi lời gọi Gemini


In [ ]:
# FIX LỖI THỰC TẾ ĐÃ GẶP: 429 RESOURCE_EXHAUSTED (free tier giới hạn 15
# request/phút/model/project). Vấn đề không chỉ ở việc BỊ giới hạn, mà ở chỗ
# rerank_one() và lvlm_answer_question()/lvlm_answer_question_multi_frame() TRƯỚC
# ĐÂY bắt Exception rồi ÂM THẦM trả về score=0.0 / answer=None — nghĩa là 1 lần dính
# 429 khiến 1 candidate ĐÚNG bị chấm "không khớp nội dung" hoặc 1 câu QA trả lời được
# bị bỏ trống, hoàn toàn oan, không phải vì nội dung sai mà vì rate limit. Chỉ riêng
# call_gemini_json() (Query Understanding) có retry — 2 hàm còn lại thì không, dù
# chúng mới là nơi tốn quota nhiều nhất (rerank gọi rerank_top_n lần/câu, QA answering
# gọi tới answer_top_n lần/câu).
#
# Giải pháp: (1) rate limiter CHỦ ĐỘNG né trước khi chạm quota thay vì đợi 429 rồi mới
# xử lý, (2) retry có backoff DÙNG CHUNG cho mọi lời gọi Gemini (Query Understanding,
# Judge, Reranker, QA answering) — không còn hàm nào "im lặng" khi dính rate limit.

import time, random, threading
from collections import deque

class GeminiRateLimiter:
    """Giới hạn số lời gọi Gemini/phút PHÍA CLIENT. Free tier: 15 request/phút/
    model/project -> để dư 1 (max=14) cho an toàn."""
    def __init__(self, max_calls_per_minute=14):
        self.max_calls = max_calls_per_minute
        self.calls = deque()
        self.lock = threading.Lock()

    def wait(self):
        with self.lock:
            now = time.time()
            while self.calls and now - self.calls[0] > 60:
                self.calls.popleft()
            if len(self.calls) >= self.max_calls:
                sleep_time = 60 - (now - self.calls[0]) + 0.5
                print(f"    [RateLimiter] {len(self.calls)} call trong 60s qua -> nghỉ {sleep_time:.1f}s")
                time.sleep(max(sleep_time, 0))
                now = time.time()
                while self.calls and now - self.calls[0] > 60:
                    self.calls.popleft()
            self.calls.append(time.time())


# Dùng CHUNG 1 rate limiter cho gemini_client_1 — vì understand_query(), rerank_one(),
# lvlm_answer_question() đều gọi CÙNG model (GEMINI_MODEL) -> cùng 1 "ví" quota, dù
# nằm ở 3 hàm khác nhau trong pipeline. gemini_client_2 (Judge, model riêng
# GEMINI_MODEL_JUDGE) dùng bộ đếm riêng vì thuộc quota metric khác.
gemini_rate_limiter = GeminiRateLimiter(max_calls_per_minute=14)
gemini_judge_rate_limiter = GeminiRateLimiter(max_calls_per_minute=14)

# FIX BUG THỰC TẾ ĐÃ GẶP: google-genai SDK có bug đã biết (GitHub issue
# googleapis/python-genai#1893 — "requests hang indefinitely (socket stall) instead of
# returning 503/Timeout") — 1 request có thể TREO VÔ THỜI HẠN, không lỗi, không trigger
# retry (vì call_gemini_with_retry() chỉ retry SAU KHI có Exception). Đặt timeout tường
# minh cho MỌI lời gọi (qua config=GenerateContentConfig(http_options=...), KHÔNG phải ở
# cấp Client() — issue #911 xác nhận đặt ở Client không đáng tin cậy) để 1 request treo
# quá lâu tự BIẾN THÀNH exception, cho retry logic có cơ hội xử lý thay vì treo mãi mãi.
GEMINI_TIMEOUT_MS = 60_000   # 60 giây/lần thử — đủ cho phần lớn request bình thường


def call_gemini_with_retry(fn, max_retries: int = 5, limiter: "GeminiRateLimiter" = None):
    """Bọc 1 lời gọi Gemini bất kỳ (truyền vào dạng lambda không tham số) với:
    rate limiter chủ động trước mỗi lần gọi + retry có backoff nếu vẫn dính
    429/503 (RESOURCE_EXHAUSTED / service quá tải). Lỗi KHÁC (không phải rate
    limit) sẽ raise ngay, không retry vô ích."""
    limiter = limiter or gemini_rate_limiter
    last_error = None
    for attempt in range(max_retries + 1):
        limiter.wait()
        try:
            return fn()
        except Exception as e:
            last_error = e
            err_str = str(e).lower()
            is_retriable = ("429" in err_str or "503" in err_str or "resource_exhausted" in err_str
                             or "timeout" in err_str or "timed out" in err_str or "deadline" in err_str)
            if is_retriable:
                wait_time = (2 ** attempt) * 2 + random.random()
                print(f"    [Retry Gemini] lần {attempt+1}/{max_retries+1} sau {wait_time:.1f}s ({e.__class__.__name__}: {str(e)[:100]})")
                time.sleep(wait_time)
                continue
            break
    raise last_error


In [ ]:
import underthesea
import re
import hashlib
import json
import time
import random

# Bộ nhớ đệm cho Query Understanding để đảm bảo tính tất định
_qu_cache = {}

QU_PROMPT_TEMPLATE = """Bạn là hệ thống phân tích truy vấn video chuyên gia cho cuộc thi AI Challenge.
Nhiệm vụ: Phân tích câu truy vấn và trích xuất thông tin dưới dạng JSON.

Câu truy vấn (đã tách từ): "{normalized_text}"

LƯU Ý ĐẶC BIỆT ĐỂ CẢI THIỆN KẾT QUẢ:
1. Suy luận tri thức (Knowledge Retrieval): Nếu câu nhắc đến các sự kiện, nhân vật, phim ảnh gián tiếp (VD: 'phim 1975 của Steven Spielberg'), bạn phải tự suy luận ra từ khóa trực quan (VD: 'Great White Shark', 'Jaws movie', 'shark swimming').
2. Mô tả hình ảnh chi tiết (Visual Scenarios): Trong field `expanded_queries`, hãy viết 3-4 câu TIẾNG ANH mô tả cụ thể những gì MẮT sẽ thấy.
   VD: 'người phụ nữ cho dê ăn' -> ["two women feeding goats in a farm", "white goat standing behind wooden fence", "woman wearing white t-shirt with red sweater on shoulders"]
3. Thực thể (Entities): Dùng dấu _ nối từ ghép tiếng Việt (VD: tàu_vũ_trụ, gỏi_cuốn_chay).

Định dạng JSON bắt buộc:
{{
  "task_type": "Textual_KIS" | "QA" | "TRAKE",
  "N": <số nguyên cho TRAKE, còn lại null>,
  "needs_temporal": <true/false>,
  "entities": [<danh từ chính>],
  "actions": [<động từ chính>],
  "attributes": [<tính từ mô tả màu sắc, trạng thái>],
  "scene": {{
    "logic": "AND" | "SEQUENCE",
    "conditions": [<danh sách object {{"entity": "...", "action": "..."}} hoặc {{"order": N, "action": "..."}} cho TRAKE>]
  }},
  "temporal_relations": <mô tả quan hệ thời gian hoặc null>,
  "question": <câu hỏi cho QA, còn lại null>,
  "expanded_queries": [<3-4 câu TIẾNG ANH mô tả hình ảnh cụ thể nhất, bao gồm cả các suy luận tri thức nếu có>],
  "audio_related": <true/false>,
  "tool_selection": ["VisualSearch", "TextSearch", "ObjectSearch", "AudioSearch"],
  "temporal_position_hint": <"start" | "middle" | "end" | null>
}}

Quy tắc task_type:
- Có chuỗi sự kiện E1, E2... hoặc thứ tự trước/sau rõ rệt -> TRAKE
- Có từ nghi vấn (Hỏi cái gì, ở đâu, là gì, bao nhiêu...) -> QA
- Mô tả một khoảnh khắc hoặc đoạn clip cụ thể -> Textual_KIS
"""

JUDGE_PROMPT_TEMPLATE = """Bạn là người kiểm tra chất lượng hệ thống phân tích truy vấn.
Câu truy vấn gốc: "{raw_text}"
Kết quả dự đoán: {predicted_json}

Kiểm tra:
1. expanded_queries có đủ chi tiết bằng tiếng Anh chưa?
2. Các thực thể ẩn đã được suy luận chưa? (VD: tên loài vật, tên phim)
3. task_type đã đúng chưa?

Trả về JSON: {{"valid": true/false, "issues": [], "corrected_fields": {{}}}}
"""

def call_gemini_json(prompt: str, client=None, model: str = None, max_retries: int = 5) -> dict:
    """FIX: dùng chung call_gemini_with_retry() (Phần 2.1) — rate limiter chủ động
    trước mỗi lần gọi, không chỉ retry SAU KHI dính lỗi như bản cũ. client_2 (Judge)
    dùng bộ đếm riêng gemini_judge_rate_limiter vì thuộc quota metric khác."""
    client = client or gemini_client_1
    model = model or GEMINI_MODEL
    limiter = gemini_judge_rate_limiter if client is gemini_client_2 else gemini_rate_limiter

    def _call():
        response = client.models.generate_content(
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.0,
                http_options=types.HttpOptions(timeout=GEMINI_TIMEOUT_MS),
            ),
        )
        return json.loads(response.text)

    try:
        return call_gemini_with_retry(_call, max_retries=max_retries, limiter=limiter)
    except Exception as e:
        raise RuntimeError(f"Lỗi Gemini JSON sau {max_retries} lần thử: {e}")

def check_with_judge(raw_text: str, predicted: dict) -> dict:
    prompt = JUDGE_PROMPT_TEMPLATE.format(raw_text=raw_text, predicted_json=json.dumps(predicted, ensure_ascii=False))
    try:
        return call_gemini_json(prompt, client=gemini_client_2, model=GEMINI_MODEL_JUDGE, max_retries=2)
    except:
        return {"valid": True, "corrected_fields": None}

def understand_query(raw_text: str, use_judge: bool = True) -> dict:
    query_hash = hashlib.md5(raw_text.strip().lower().encode()).hexdigest()
    if query_hash in _qu_cache: return _qu_cache[query_hash].copy()

    tokens = underthesea.word_tokenize(raw_text)
    normalized_text = " ".join(tokens)
    prompt = QU_PROMPT_TEMPLATE.format(normalized_text=normalized_text)
    result = call_gemini_json(prompt)

    if result.get("task_type") == "TRAKE":
        numbered = re.findall(r'[Ee](\d+)', raw_text)
        if numbered: result["N"] = len(set(numbered))
        result["needs_temporal"] = True

    if use_judge:
        judge = check_with_judge(raw_text, result)
        if not judge.get("valid", True) and judge.get("corrected_fields"):
            result.update(judge["corrected_fields"])

    result["raw_text"] = raw_text
    result["normalized_text"] = normalized_text
    _qu_cache[query_hash] = result.copy()
    return result

---
## Phần 4 — Visual Search

**Giải thích:** encode `expanded_queries` bằng CLIP Text Encoder, so sánh cosine similarity với FAISS Index — đúng cơ chế đã minh họa bằng dữ liệu thật (`L30_V036.npy`) trước đó. Kết quả trả về vị trí trong FAISS, cần map ngược lại thành `(video_id, frame_id thật)`.

In [78]:
def get_real_frame_id(video_id: str, local_frame_idx: int) -> str:
    """Đổi local_frame_idx (0-based) sang frame_id thật.
    SỬA LỖI: Cột 'n' trong map_keyframes_df bắt đầu từ 1,
    nên phải tìm n == local_frame_idx + 1."""
    if local_frame_idx is None or pd.isna(local_frame_idx):
        return None

    row = map_keyframes_df[
        (map_keyframes_df["video_id"] == video_id) &
        (map_keyframes_df["n"] == int(local_frame_idx) + 1)
    ]
    if len(row) > 0:
        return int(row.iloc[0]["frame_idx"])
    return None

In [79]:
def visual_search(qu_result: dict, top_k: int = 100) -> pd.DataFrame:
    all_results = []
    skipped_no_mapping = 0

    for query_text in qu_result["expanded_queries"]:
        query_vector = clip_model.encode(query_text).astype("float32").reshape(1, -1)
        faiss.normalize_L2(query_vector)   # phải chuẩn hóa GIỐNG cách đã làm lúc build index

        scores, indices = clip_index.search(query_vector, top_k)

        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            row = clip_mapping_df.iloc[idx]
            frame_id = get_real_frame_id(row["video_id"], row["local_frame_idx"])
            if frame_id is None:
                skipped_no_mapping += 1
                if skipped_no_mapping <= 5:   # in vài mẫu đầu để điều tra, tránh spam
                    print(f"    [DEBUG mapping fail] video_id={row['video_id']}, "
                          f"local_frame_idx={row['local_frame_idx']}")
                continue   # KHÔNG đưa vào kết quả nếu không map được frame_id thật
            all_results.append({
                "video_id": row["video_id"],
                "frame_id": frame_id,
                "local_frame_idx": row["local_frame_idx"],   # giữ lại để Reranker/TRAKE load ảnh
                "score": float(score),
                "module": "visual",
            })

    if skipped_no_mapping > 0:
        print(f"  [CẢNH BÁO] Visual Search: {skipped_no_mapping} candidate bị bỏ qua vì "
              f"không map được frame_id thật -> kiểm tra lại get_real_frame_id() / map_keyframes_df")

    df = pd.DataFrame(all_results)
    if len(df) == 0:
        return df
    df = df.sort_values("score", ascending=False).drop_duplicates(["video_id", "frame_id"])
    return df.head(top_k).reset_index(drop=True)

---
## Phần 5 — Text Search (BM25 trên OCR + ASR)

**Giải thích:** dùng `entities` + `actions` làm từ khóa, tra BM25 Index đã build. Với 3 loại truy vấn sơ tuyển, module này thường trả về ít/rỗng nếu câu không liên quan chữ viết/lời nói — điều đó BÌNH THƯỜNG, không phải lỗi.

In [ ]:
import subprocess

_fps_cache = {}   # cache fps theo video_path, chỉ dùng cho fallback ffprobe
_video_fps_lookup = (
    map_keyframes_df.groupby("video_id")["fps"].first().to_dict()
    if "fps" in map_keyframes_df.columns else {}
)


def get_video_fps(video_id: str, video_path: str = None) -> float:
    """Ưu tiên đọc fps TRỰC TIẾP từ map_keyframes_df (đã có sẵn cột 'fps', nhanh,
    đáng tin) — chỉ fallback sang ffprobe (chậm hơn, cần đọc file video, và cần
    find_video_file() đã định nghĩa) nếu video_id không có trong bảng."""
    if video_id in _video_fps_lookup:
        return float(_video_fps_lookup[video_id])

    if video_path is None:
        video_path = find_video_file(video_id)
    if video_path is None:
        return 25.0
    if video_path in _fps_cache:
        return _fps_cache[video_path]
    try:
        cmd = f'ffprobe -v error -select_streams v:0 -show_entries stream=r_frame_rate -of csv=p=0 "{video_path}"'
        output = subprocess.check_output(cmd, shell=True).decode().strip()
        num, denom = output.split('/')
        fps = float(num) / float(denom)
    except Exception:
        fps = 25.0
    _fps_cache[video_path] = fps
    return fps


def asr_timestamp_to_frame_id(video_id: str, timestamp_sec: float):
    """Quy đổi timestamp (giây) của ASR sang frame_id thật, dùng fps THẬT — giờ đọc
    trực tiếp từ map_keyframes_df, KHÔNG cần find_video_file() cho trường hợp phổ
    biến (video_id đã có trong bảng)."""
    if timestamp_sec is None:
        return None
    fps = get_video_fps(video_id)
    return int(timestamp_sec * fps)


def text_search(qu_result: dict, top_k: int = 100) -> pd.DataFrame:
    if bm25_index is None:
        return pd.DataFrame(columns=["video_id", "frame_id", "local_frame_idx", "score", "module"])

    search_terms = qu_result["entities"] + qu_result["actions"]
    if not search_terms:
        return pd.DataFrame(columns=["video_id", "frame_id", "local_frame_idx", "score", "module"])

    tokenized_query = " ".join(search_terms).lower().split()
    scores = bm25_index.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    skipped_no_mapping = 0

    for idx in top_indices:
        if scores[idx] <= 0:
            continue
        row = text_index_df.iloc[idx]

        if row.get("source") == "RECAP":
            # THÊM: candidate mới từ ReCap Captioning (Offline Phần 10) — caption có trí nhớ
            # ngữ cảnh xuyên suốt video, tìm ra được cả những frame mà bản thân hình ảnh KHÔNG
            # đủ đặc trưng để Visual Search tìm ra (VD màn hình chuyển cảnh gần như trống, chỉ
            # còn "biết" nhờ ngữ cảnh chữ đã lưu từ trước).
            start_f, end_f = row.get("start_frame"), row.get("end_frame")
            if pd.isna(start_f) or pd.isna(end_f):
                skipped_no_mapping += 1
                continue
            frame_id = int((float(start_f) + float(end_f)) / 2)   # điểm giữa khoảng shot
            recap_local_idx = get_local_frame_idx_from_frame_id(row["video_id"], frame_id)
            results.append({
                "video_id": row["video_id"], "frame_id": frame_id, "local_frame_idx": recap_local_idx,
                "score": float(scores[idx]), "module": "text_recap",
            })
            continue

        if row.get("source") == "ASR":
            mid_timestamp = None
            if not pd.isna(row.get("start")) and not pd.isna(row.get("end")):
                mid_timestamp = (float(row["start"]) + float(row["end"])) / 2
            frame_id = asr_timestamp_to_frame_id(row["video_id"], mid_timestamp)
            if frame_id is None:
                skipped_no_mapping += 1
                continue
            # ĐÃ SỬA: trước đây local_frame_idx LUÔN None cho ASR (không xác thực
            # được, không tra được ảnh) — giờ tra ngược qua map_keyframes_df, giống
            # cách đã sửa cho Neighbor Expansion, để candidate ASR cũng hiển thị/
            # đánh giá được như candidate OCR.
            asr_local_idx = get_local_frame_idx_from_frame_id(row["video_id"], frame_id)
            results.append({
                "video_id": row["video_id"], "frame_id": frame_id, "local_frame_idx": asr_local_idx,
                "score": float(scores[idx]), "module": "text_asr",
            })
            continue

        if pd.isna(row.get("frame_index")):
            continue
        local_idx = int(row["frame_index"])
        frame_id = get_real_frame_id(row["video_id"], local_idx)
        if frame_id is None:
            skipped_no_mapping += 1
            if skipped_no_mapping <= 5:
                print(f"    [DEBUG mapping fail] video_id={row['video_id']}, local_frame_idx={local_idx}")
            continue

        results.append({
            "video_id": row["video_id"], "frame_id": frame_id, "local_frame_idx": local_idx,
            "score": float(scores[idx]), "module": "text_ocr",
        })

    if skipped_no_mapping > 0:
        print(f"  [CẢNH BÁO] Text Search: {skipped_no_mapping} candidate bị bỏ qua vì không map được frame_id thật")

    return pd.DataFrame(results)


_video_max_frame_lookup = (
    map_keyframes_df.groupby("video_id")["frame_idx"].max().to_dict()
    if "frame_idx" in map_keyframes_df.columns else {}
)


def clip_frame_number(video_id: str, frame_number: int) -> int:
    """Giới hạn frame_number trong phạm vi hợp lý của video. ƯU TIÊN dùng
    length (giây, từ Media-info) x fps -> cận trên CHÍNH XÁC hơn nhiều so với
    chỉ dựa vào keyframe cuối (có thể sớm hơn thực tế 1 chút). Fallback về
    frame_idx lớn nhất trong map_keyframes_df nếu video không có length hợp lệ.
    ĐẶT Ở ĐÂY (Phần 5, ngay sau get_video_fps) thay vì Phần 10 như trước — vì
    add_neighbor_expansion() ở Phần 8 cần gọi hàm này, và Phần 8 chạy TRƯỚC
    Phần 10 trong notebook -> để ở Phần 10 gây lỗi 'NameError: chưa định nghĩa'."""
    if frame_number is None:
        return None

    upper_bound = None
    # FIX: cột thật trong media_full.csv (nguồn: Media_Info_Exploitation.ipynb) tên là
    # "length_seconds", KHÔNG PHẢI "length" — bản trước check sai tên cột nên điều kiện này
    # luôn False, khiến cải tiến "giới hạn frame TRAKE bằng thời lượng thật" chưa từng thực sự
    # chạy, âm thầm rơi về fallback cũ kém chính xác hơn (khớp đúng vấn đề đã gặp: Event 4
    # trích ảnh thất bại vì cận trên ước lượng thấp hơn thực tế).
    if len(media_info_df) > 0 and "length_seconds" in media_info_df.columns:
        row = media_info_df[media_info_df["video_id"] == video_id]
        if len(row) > 0 and pd.notna(row.iloc[0]["length_seconds"]):
            fps = get_video_fps(video_id)
            if fps > 0:
                upper_bound = int(float(row.iloc[0]["length_seconds"]) * fps)

    if upper_bound is None:
        upper_bound = _video_max_frame_lookup.get(video_id)

    if upper_bound is None:
        return max(0, frame_number)
    return max(0, min(frame_number, upper_bound))

---
## Phần 6 — Object/Metadata Search

**Giải thích:** lọc `objects_df` theo nhãn vật thể, VÀ dùng thêm `media_info_df`
(title/description video) làm **filter tùy chọn** nếu câu query có gợi ý chủ đề/từ khóa
khớp với metadata — trước đây `media_info_df` được load ở Phần 1 nhưng chưa từng dùng, giờ tận dụng.

In [ ]:
# Bảng ánh xạ Việt -> Anh (nhãn OpenImages V4) — ĐÃ ĐỐI CHIẾU VỚI DATA THẬT (P3, chạy trên
# objects_index.parquet thật — 2.054.901 dòng, 545 nhãn duy nhất, 873 video):
# - Xoá 25 entry KHÔNG tồn tại (OpenImages V4 chỉ nhận diện VẬT THỂ cụ thể, không nhận diện
#   địa danh/khung cảnh — "biển", "núi", "chùa", "khách_sạn"... không có nhãn tương ứng, đây
#   là giới hạn công nghệ chứ không phải thiếu sót cấu hình). Toàn bộ nhóm DU LỊCH (Phần 5 cũ)
#   không có nhãn nào tồn tại thật -> đã bỏ hẳn, nên dựa vào Visual Search cho domain này.
# - Món ăn CỤ THỂ (canh/lẩu/thịt bò/gạo...) cũng không tồn tại — OpenImages chỉ có nhãn
#   "Food" chung chung, không phân biệt món.
# - Remap 2 chỗ theo đúng nhãn thật gần nghĩa nhất: "ly/cốc" -> Mug (không phải Cup — Cup
#   không tồn tại, có Coffee cup/Mug/Wine glass), "mì/bún" -> Pasta (không có Noodle).
# - Bổ sung nhãn PHỔ BIẾN NHẤT thật sự có trong data mà trước đây CHƯA có từ tiếng Việt nào
#   trỏ tới — đặc biệt "Clothing" (230.781 lượt — SAU "Person", đây là nhãn nhiều thứ 2 toàn
#   bộ dataset!) và "Human face" (139.757 lượt) — bỏ sót 2 nhãn này gần như bỏ phí nửa lượng
#   detection có sẵn.
#
# CẬP NHẬT (phát hiện qua test thật với câu "hai tay chạm mũi chân"): "nón" (từ đồng
# nghĩa vùng miền của "mũ") và "tay" (Human hand — 36.878 lượt, khá phổ biến) từng
# BỊ THIẾU dù có nhãn thật tương ứng — khiến Object Search bỏ lỡ đúng chi tiết quan
# trọng nhất câu hỏi, để Visual Search một mình gánh hết.
VI_TO_EN_LABEL = {
    # ═══ NHÓM 1 — NẤU ĂN (ưu tiên CAO NHẤT, ~57% dataset — ViVU TV) ═══
    # Dụng cụ bếp
    "dao": "Kitchen knife", "con_dao": "Kitchen knife",
    "thớt": "Cutting board",
    "chảo": "Frying pan",
    "bát": "Bowl", "tô": "Bowl",
    "đĩa": "Plate", "dĩa": "Plate",
    "đũa": "Chopsticks",
    "muỗng": "Spoon", "thìa": "Spoon",
    "nĩa": "Fork",
    "ly": "Mug", "cốc": "Mug",                    # SỬA: Cup không tồn tại, Mug có thật
    "ly_rượu": "Wine glass",                       # THÊM: nhãn thật riêng cho ly rượu
    "tách_cà_phê": "Coffee cup",                   # THÊM: nhãn thật riêng cho tách cà phê
    "bếp": "Gas stove", "bếp_gas": "Gas stove",
    "lò": "Oven", "lò_nướng": "Oven",
    "tủ_lạnh": "Refrigerator",
    "ấm": "Kettle", "ấm_đun_nước": "Kettle",       # THÊM: Kettle tồn tại thật
    # Nguyên liệu / món ăn — CHỈ giữ nhãn THẬT SỰ tồn tại (đã xoá thịt/gạo/canh/gia vị...)
    "tôm": "Shrimp", "cá": "Fish", "hải_sản": "Seafood",
    "rau": "Vegetable", "rau_củ": "Vegetable",
    "cà_chua": "Tomato",                           # THÊM: 17.406 lượt, rất phổ biến
    "trái_cây": "Fruit", "hoa_quả": "Fruit",
    "trứng": "Egg",
    "mì": "Pasta", "bún": "Pasta",                 # SỬA: Noodle không tồn tại, Pasta có thật
    "bánh": "Cake", "bánh_mì": "Bread",
    "salad": "Salad", "rau_sống": "Salad",
    "tráng_miệng": "Dessert",                      # THÊM: 8.921 lượt
    "đồ_uống": "Drink", "thức_uống": "Drink",      # THÊM
    "đồ_ăn_vặt": "Snack",                          # THÊM

    # ═══ NHÓM 2 — TIN TỨC (ưu tiên CAO, ~28% dataset — Tuổi Trẻ/Thanh Niên/60 Giây) ═══
    "người": "Person", "đàn_ông": "Man", "phụ_nữ": "Woman",
    "quần_áo": "Clothing", "áo_quần": "Clothing",  # THÊM: 230.781 lượt — nhãn phổ biến THỨ 2
                                                     # toàn dataset, chỉ sau "Person"
    "khuôn_mặt": "Human face", "mặt_người": "Human face",  # THÊM: 139.757 lượt
    "bé_trai": "Boy", "bé_gái": "Girl",            # THÊM
    "diễn_giả": "Person", "người_dẫn_chương_trình": "Person",
    "phóng_viên": "Person", "cảnh_sát": "Person", "sĩ_quan": "Person",
    "micro": "Microphone", "máy_quay": "Camera", "máy_ảnh": "Camera",
    "tivi": "Television", "màn_hình": "Television",
    "áo_vest": "Suit", "cà_vạt": "Tie", "kính": "Glasses",
    "bàn": "Table", "ghế": "Chair", "sách": "Book",
    "laptop": "Laptop", "điện_thoại": "Mobile phone",
    "chai": "Bottle",                              # THÊM

    # ═══ NHÓM 3 — THỂ THAO (~8% dataset — HTV Sports) ═══
    "vận_động_viên": "Person",
    "bóng": "Ball", "quả_bóng": "Ball",
    "xe_đạp": "Bicycle", "mũ_bảo_hiểm": "Helmet",
    "trống": "Drum",
    # ĐÃ XOÁ: "lân"->Costume, "huy_chương"->Medal, "cúp"->Trophy (không tồn tại nhãn thật)

    # ═══ NHÓM 4 — THIÊN NHIÊN/ĐỘNG VẬT (~5% dataset — HTV Entertainment) ═══
    "ong": "Bee", "tổ_ong": "Beehive",
    "động_vật": "Animal",
    "chó": "Dog", "mèo": "Cat", "chim": "Bird", "gà": "Chicken",
    "rừng": "Tree", "cây": "Tree", "cây_xanh": "Tree",
    "thuyền": "Boat", "ghe": "Boat",
    # ĐÃ XOÁ: "chồn"->Animal (trùng, gộp vào "động_vật"), "sông"->River, "ruộng"->Field,
    # "nông_trại"->Farm (không tồn tại nhãn thật)

    # ĐÃ XOÁ HẲN NHÓM 5 — DU LỊCH: "chùa/đền"->Temple, "núi"->Mountain, "biển"->Sea,
    # "bãi_biển"->Beach, "khách_sạn"->Hotel — KHÔNG nhãn nào trong nhóm này tồn tại thật.
    # OpenImages V4 không nhận diện địa danh/khung cảnh — domain này nên dựa vào Visual
    # Search (CLIP), không phải Object Search.

    # ═══ CHUNG (dùng được cho mọi domain) ═══
    "nhà": "House", "cửa": "Door", "cửa_sổ": "Window",
    "xe": "Car", "xe_hơi": "Car", "xe_máy": "Motorcycle",
    "giày": "Footwear", "mũ": "Hat", "nón": "Hat", "túi_xách": "Handbag",
    "tay": "Human hand", "bàn_tay": "Human hand",
    "hoa": "Flower", "đồng_hồ": "Clock",
    "đồ_chơi": "Toy",                              # THÊM
    "bánh_xe": "Wheel",                            # THÊM
    "tòa_nhà": "Building",                         # THÊM
    "áo": "Clothing",                              # THÊM: dạng đơn của quần_áo, giúp khớp
                                                     # component cho từ ghép (VD "áo_xanh_lá")
}

# THÊM (phát hiện qua test thật — câu "đàn sư tử"/"mực xào"): "Lion" và "Squid" đều là
# nhãn thật tồn tại nhưng bị bỏ sót hoàn toàn.
VI_TO_EN_LABEL["sư_tử"] = "Lion"
VI_TO_EN_LABEL["mực"] = "Squid"
VI_TO_EN_LABEL["con_vật"] = "Animal"   # đồng nghĩa với "động_vật" đã có — Gemini dùng cả 2 cách nói
VI_TO_EN_LABEL["kéo"] = "Scissors"     # THÊM (test thật, câu cắt chùm nho)


def get_metadata_boost_videos(qu_result: dict) -> set:
    """Dùng media_info_df (title + description + keywords + author) làm BỘ LỌC BỔ SUNG —
    trả về tập video_id khớp với 1 trong các entity/action của câu query. KHÔNG bắt buộc,
    chỉ dùng để boost điểm nhẹ — không loại video nào ra chỉ vì không khớp.

    FIX: "keywords" đọc từ media_full.csv (Media_Info_Exploitation.ipynb) LUÔN là 1 chuỗi
    thường (đã gộp từ list bằng dấu cách trước khi ghi CSV) — không còn là list trong bộ nhớ.
    Bản trước vẫn check isinstance(kws, list), luôn False sau khi qua CSV -> keywords bị vô
    hiệu hoá ÂM THẦM dù code trông như đang dùng nó (không lỗi, không cảnh báo). Giờ dùng
    thẳng dạng string.

    THÊM: "author" (tên kênh, VD "60 Giây Official", "ViVU TV") trước đây CHỈ được dùng
    để PHẠT ở reciprocal_rank_fusion() (kênh chiếm >30% index bị giảm điểm), chưa từng được
    dùng để BOOST — dù câu query nêu đúng tên chương trình/kênh (VD "trong bản tin 60 Giây...")
    là tín hiệu rất mạnh để xác định đúng video. Giờ ghép thêm "author" vào text_cols để tận
    dụng tín hiệu này.
    """
    if len(media_info_df) == 0:
        return set()
    search_terms = [w.replace("_", " ") for w in qu_result.get("entities", []) + qu_result.get("actions", [])]
    if not search_terms:
        return set()

    matched = set()
    keywords_col = media_info_df["keywords"].fillna("") if "keywords" in media_info_df.columns else ""
    author_col = media_info_df["author"].fillna("") if "author" in media_info_df.columns else ""
    text_cols = (media_info_df["title"].fillna("") + " " + media_info_df["description"].fillna("")
                 + " " + keywords_col + " " + author_col)
    for kw in search_terms:
        mask = text_cols.str.contains(kw, case=False, na=False, regex=False)
        matched.update(media_info_df.loc[mask, "video_id"].tolist())
    return matched


def _normalize_vi_key(s: str) -> str:
    """Chuẩn hoá 1 chuỗi tiếng Việt trước khi so khớp với VI_TO_EN_LABEL — bỏ khoảng
    trắng thừa, hạ chữ thường, đồng nhất khoảng trắng/underscore. FIX: trước đây so khớp
    string y hệt tuyệt đối — nếu Gemini trích entity ra dạng "con dao" (khoảng trắng)
    trong khi dict chỉ có key "con_dao" (underscore), match thất bại HOÀN TOÀN dù đúng
    nghĩa, làm Object Search mất tín hiệu một cách âm thầm."""
    return s.strip().lower().replace(" ", "_")


_VI_TO_EN_LABEL_NORMALIZED = {_normalize_vi_key(k): v for k, v in VI_TO_EN_LABEL.items()}


def _match_vi_label(entity: str):
    """Tìm nhãn Anh khớp với 1 entity — thử khớp CHÍNH XÁC trước (sau chuẩn hoá). Nếu
    không khớp, thử khớp THEO THÀNH PHẦN: Gemini hay trích entity dạng GHÉP nhiều từ (VD
    "đàn_sư_tử", "bếp_lửa", "áo_xanh_lá") trong khi dict chỉ có từ ĐƠN/cụm ngắn ("sư_tử",
    "bếp", "áo") — kiểm tra xem có KEY nào trong dict là tập con TRỌN VẸN các từ của entity
    hay không (không so khớp substring thô, tránh khớp nhầm kiểu "áy" chứa "áo"). Ưu tiên
    key khớp DÀI NHẤT (nhiều từ nhất) nếu có nhiều khả năng, để đặc hiệu hơn."""
    norm = _normalize_vi_key(entity)
    if norm in _VI_TO_EN_LABEL_NORMALIZED:
        return _VI_TO_EN_LABEL_NORMALIZED[norm]

    entity_parts = set(norm.split("_"))
    best_label, best_len = None, 0
    for key, label in _VI_TO_EN_LABEL_NORMALIZED.items():
        key_parts = key.split("_")
        if all(p in entity_parts for p in key_parts) and len(key_parts) > best_len:
            best_label, best_len = label, len(key_parts)
    return best_label


def object_search(qu_result: dict, top_k: int = 100) -> pd.DataFrame:
    en_labels = []
    for e in qu_result["entities"]:
        matched = _match_vi_label(e)
        if matched:
            en_labels.append(matched)
    if not en_labels:
        return pd.DataFrame(columns=["video_id", "frame_id", "local_frame_idx", "score", "module"])

    matched = objects_df[objects_df["label"].isin(en_labels)]

    grouped = matched.groupby(["video_id", "frame_index"]).agg(
        n_matched_labels=("label", "nunique"),
        avg_confidence=("confidence", "mean"),
    ).reset_index()

    grouped["score"] = grouped["n_matched_labels"] * grouped["avg_confidence"]

    boost_videos = get_metadata_boost_videos(qu_result)
    if boost_videos:
        grouped.loc[grouped["video_id"].isin(boost_videos), "score"] *= 1.2

    grouped = grouped.sort_values("score", ascending=False).head(top_k * 2)

    results = []
    skipped_no_mapping = 0
    for _, row in grouped.iterrows():
        local_idx = int(row["frame_index"])
        frame_id = get_real_frame_id(row["video_id"], local_idx)
        if frame_id is None:
            skipped_no_mapping += 1
            continue
        results.append({
            "video_id": row["video_id"], "frame_id": frame_id, "local_frame_idx": local_idx,
            "score": row["score"], "module": "object",
        })

    if skipped_no_mapping > 0:
        print(f"  [CẢNH BÁO] Object Search: {skipped_no_mapping} candidate bị bỏ qua vì không map được frame_id thật")

    return pd.DataFrame(results).head(top_k).reset_index(drop=True)

### 6.1 — Xác nhận THẬT: VI_TO_EN_LABEL có khớp nhãn thật trong objects_df không?

**Giải thích:** `VI_TO_EN_LABEL` (cell trên) là bảng ĐỀ XUẤT dựa trên suy đoán tên nhãn OpenImages V4 phổ biến — CHƯA từng được đối chiếu với dữ liệu thật. Nếu 1 nhãn tiếng Anh trong dict không tồn tại thật trong `objects_df`, mọi từ tiếng Việt trỏ tới nó sẽ **luôn cho 0 kết quả** ở Object Search — âm thầm, không báo lỗi gì. Cell này đối chiếu trực tiếp với dữ liệu Offline thật vừa xong, in rõ nhãn nào khớp/không khớp, và liệt kê top nhãn phổ biến thật sự trong dataset để bổ sung nếu thiếu.


In [ ]:
real_labels = set(objects_df["label"].unique())
print(f"Tổng số nhãn (label) THẬT có trong objects_df: {len(real_labels)}\n")

valid_entries, invalid_entries = {}, {}
for vi, en in VI_TO_EN_LABEL.items():
    (valid_entries if en in real_labels else invalid_entries)[vi] = en

print(f"[OK]    Nhãn HỢP LỆ (khớp label thật): {len(valid_entries)}/{len(VI_TO_EN_LABEL)}")
print(f"[THIẾU] Nhãn KHÔNG khớp label nào (Object Search sẽ luôn 0 kết quả): "
      f"{len(invalid_entries)}/{len(VI_TO_EN_LABEL)}")

if invalid_entries:
    print("\nDanh sách entry bị vô hiệu hoá (từ tiếng Việt -> nhãn Anh KHÔNG tồn tại thật):")
    for vi, en in sorted(invalid_entries.items()):
        print(f"  '{vi}' -> '{en}'  (KHÔNG tìm thấy trong objects_df)")

print("\nTop 30 nhãn PHỔ BIẾN NHẤT thật sự có trong objects_df (tần suất xuất hiện):")
print(objects_df["label"].value_counts().head(30))

print("\n=> Đối chiếu 2 danh sách trên: nếu 1 nhãn phổ biến (top 30) chưa có mặt trong")
print("   VI_TO_EN_LABEL, cân nhắc thêm entry tiếng Việt tương ứng ở cell trên. Nhãn nào bị")
print("   liệt kê ở mục [THIẾU] phía trên, cân nhắc sửa/xoá vì hiện KHÔNG dùng được.")


---
## Phần 6b — Audio Search (bổ sung — cấu trúc sẵn sàng, chờ Offline hoàn thiện Audio Embedding)

**Giải thích:** module này trước đây CHƯA TỪNG có code — giờ viết đầy đủ cấu trúc, nhưng Offline vẫn chưa build Audio Embedding Index (đang ưu tiên thấp nhất). Nên hiện tại module này **luôn trả về rỗng** một cách AN TOÀN (không lỗi), tự động kích hoạt khi Offline hoàn thiện Phần 4c.

In [ ]:
def load_audio_index():
    """Tải Audio Embedding FAISS Index nếu đã có (từ Offline Phần 8) — nếu chưa
    có, trả về None, KHÔNG làm gãy pipeline (giống cách xử lý BM25 ở Phần 1)."""
    idx_path = os.path.join(EXTRACTED_ROOT, "audio_embedding", "audio_faiss.index")
    map_path = os.path.join(EXTRACTED_ROOT, "audio_embedding", "audio_mapping.parquet")
    if os.path.exists(idx_path) and os.path.exists(map_path):
        return faiss.read_index(idx_path), pd.read_parquet(map_path)
    return None, None


audio_index, audio_mapping_df = load_audio_index()
clap_model = None

if audio_index is None:
    print("Audio Embedding Index CHƯA có (Offline Phần 8 chưa chạy) -> Audio Search sẽ luôn trả về rỗng.")
else:
    print(f"Audio Embedding Index: {audio_index.ntotal} vector -> nạp CLAP model để encode query...")
    # CHỈ pip install + load CLAP (khá nặng) khi THỰC SỰ có index để dùng — tránh làm chậm
    # notebook cho người chưa chạy Offline Phần 8, giữ đúng tinh thần "tùy chọn, không bắt buộc".
    # Dùng transformers.ClapModel (KHÔNG dùng gói PyPI "laion-clap" riêng) — gói đó ép hạ
    # numpy xuống 1.26.4, xung đột với hàng loạt thư viện khác đang chạy trong Online
    # (đúng bug đã gặp bên Offline). transformers tương thích numpy 2.x, an toàn hơn.
    os.system("pip install -U transformers --quiet")
    import torch
    from transformers import ClapModel, ClapProcessor
    CLAP_CHECKPOINT = "laion/clap-htsat-unfused"
    clap_device = "cuda" if torch.cuda.is_available() else "cpu"
    clap_model = ClapModel.from_pretrained(CLAP_CHECKPOINT).to(clap_device)
    clap_processor = ClapProcessor.from_pretrained(CLAP_CHECKPOINT)
    clap_model.eval()
    print(f"Đã load CLAP model qua transformers ({clap_device}) — Audio Search giờ hoạt động đầy đủ.")

def _extract_clap_features(output):
    """transformers bản mới đang chuẩn hoá lại get_audio_features()/get_text_features()
    -> có thể trả về BaseModelOutputWithPooling thay vì tensor thuần như trước (xem GitHub
    issue huggingface/transformers#42401). Lấy pooler_output nếu có, fallback về chính
    object nếu không (coi như tensor thuần — tương thích ngược với bản cũ)."""
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output
    return output



In [ ]:
def audio_search(qu_result: dict, top_k: int = 100) -> pd.DataFrame:
    """Audio Search — chỉ chạy nếu (1) có Audio Embedding Index VÀ (2) LLM xác định câu
    query thực sự liên quan âm thanh (audio_related=true). Dùng CLAP Text Encoder để encode
    câu query vào CHUNG không gian vector với audio embedding đã build sẵn ở Offline Phần 8
    (giống cách CLIP Text Encoder dùng cho visual_search()). Mỗi vector trong audio_index
    ứng với 1 CỬA SỔ THỜI GIAN (start_sec/end_sec), không phải 1 keyframe cụ thể — nên quy
    đổi sang frame_id thật bằng đúng cơ chế đã dùng cho ASR (asr_timestamp_to_frame_id)."""
    if audio_index is None or clap_model is None or not qu_result.get("audio_related", False):
        return pd.DataFrame(columns=["video_id", "frame_id", "local_frame_idx", "score", "module"])

    query_text = qu_result.get("question") or " ".join(qu_result.get("expanded_queries", []))
    if not query_text:
        return pd.DataFrame(columns=["video_id", "frame_id", "local_frame_idx", "score", "module"])

    clap_inputs = clap_processor(text=[query_text], return_tensors="pt", padding=True)
    clap_inputs = {k: v.to(clap_device) for k, v in clap_inputs.items()}
    with torch.no_grad():
        text_out = clap_model.get_text_features(**clap_inputs)
        query_vector = _extract_clap_features(text_out).cpu().numpy().astype("float32")
    faiss.normalize_L2(query_vector)   # phải chuẩn hóa GIỐNG cách đã làm lúc build audio_index

    scores, indices = audio_index.search(query_vector, top_k)

    results = []
    skipped_no_mapping = 0
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        row = audio_mapping_df.iloc[idx]
        mid_timestamp = (float(row["start_sec"]) + float(row["end_sec"])) / 2
        frame_id = asr_timestamp_to_frame_id(row["video_id"], mid_timestamp)
        if frame_id is None:
            skipped_no_mapping += 1
            continue
        local_idx = get_local_frame_idx_from_frame_id(row["video_id"], frame_id)
        results.append({
            "video_id": row["video_id"], "frame_id": frame_id, "local_frame_idx": local_idx,
            "score": float(score), "module": "audio",
        })

    if skipped_no_mapping > 0:
        print(f"  [CẢNH BÁO] Audio Search: {skipped_no_mapping} candidate bị bỏ qua vì không map được frame_id thật")

    df = pd.DataFrame(results)
    if len(df) == 0:
        return df
    return df.sort_values("score", ascending=False).drop_duplicates(["video_id", "frame_id"]).head(top_k).reset_index(drop=True)


---
## Phần 7 — Candidate Fusion (Normalize scores + RRF + Adaptive weights + Dedup)

**Giải thích:** đủ cả 4 bước đã thiết kế:
- **Normalize scores**: đưa điểm mỗi module về khoảng [0,1] (min-max) — vì Visual (cosine ~0.2-0.4), Text (BM25 không giới hạn trên), Object (0-1) có thang đo khác hẳn nhau.
- **RRF**: kết hợp theo THỨ HẠNG (không phụ thuộc giá trị điểm tuyệt đối — vẫn đáng tin cậy dù không normalize, nhưng làm rõ ràng, nhất quán hơn khi có normalize trước).
- **Adaptive weights**: trọng số động theo đặc điểm câu query.
- **Dedup**: tự động qua key `(video_id, frame_id)` khi gộp vào `rrf_scores` dict.

In [ ]:
def normalize_scores(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize scores — đưa cột 'score' về khoảng [0,1] bằng min-max scaling."""
    if len(df) == 0:
        return df
    df = df.copy()
    min_s, max_s = df["score"].min(), df["score"].max()
    df["score"] = (df["score"] - min_s) / (max_s - min_s) if max_s > min_s else 1.0
    return df

def reciprocal_rank_fusion(named_result_dfs: dict, weights: dict = None, k: int = 60) -> pd.DataFrame:
    """RRF với cải tiến Source-Aware: Giảm nhẹ điểm của các nguồn có mật độ keyframe quá dày
    để tránh việc một kênh (như Báo Thanh Niên) áp đảo kết quả chỉ vì số lượng.

    THÊM: theo dõi "modules" — tập hợp các module (visual/text/object/audio) ĐỘC LẬP cùng
    tìm ra 1 candidate (video_id, frame_id). Dùng làm tín hiệu "đồng thuận" để đánh giá độ
    tin cậy kết quả cuối — nhiều module độc lập cùng đồng ý đáng tin hơn chỉ 1 module tìm ra."""
    rrf_scores = {}
    extra_info = {}
    source_modules = {}   # key (video_id, frame_id) -> set các module đã tìm ra candidate này

    # Tính toán trọng số trừng phạt nguồn (Source Penalty) nếu cần
    source_counts = clip_mapping_df['video_id'].map(media_info_df.set_index('video_id')['author']).value_counts(normalize=True)

    for name, df in named_result_dfs.items():
        if df is None or len(df) == 0:
            continue
        df = normalize_scores(df)
        w = (weights or {}).get(name, 1.0)
        df_sorted = df.sort_values("score", ascending=False).reset_index(drop=True)

        for rank, row in df_sorted.iterrows():
            key = (row["video_id"], row["frame_id"])

            # Source-Aware Adjustment: Nếu video thuộc nguồn chiếm > 30% index, giảm nhẹ ưu thế rank
            author = media_info_df.loc[media_info_df['video_id'] == row['video_id'], 'author'].values
            penalty = 0.85 if len(author) > 0 and source_counts.get(author[0], 0) > 0.3 else 1.0

            rrf_scores[key] = rrf_scores.get(key, 0.0) + (w * penalty * (1.0 / (k + rank + 1)))

            if key not in extra_info and "local_frame_idx" in row.index:
                extra_info[key] = row["local_frame_idx"]

            source_modules.setdefault(key, set()).add(name)

    if not rrf_scores:
        return pd.DataFrame(columns=["video_id", "frame_id", "rrf_score", "local_frame_idx", "modules"])

    fused = pd.DataFrame([
        {"video_id": vid, "frame_id": fid, "rrf_score": score, "local_frame_idx": extra_info.get((vid, fid)),
         "modules": sorted(source_modules.get((vid, fid), set()))}
        for (vid, fid), score in rrf_scores.items()
    ])
    return fused.sort_values("rrf_score", ascending=False).reset_index(drop=True)

def compute_adaptive_weights(qu_result: dict) -> dict:
    """Tính toán trọng số động cho từng module dựa trên kết quả Query Understanding."""
    weights = {"visual": 0.0, "text": 0.0, "object": 0.0, "audio": 0.0}

    # Base weights
    base_weight_visual = 1.0
    base_weight_text = 0.8
    base_weight_object = 0.9
    base_weight_audio = 0.7  # Audio Search ĐÃ implement đầy đủ (Offline Phần 8 + CLAP Text
                          # Encoder ở Online) — giá trị 0.7 là trọng số khởi điểm, có thể
                          # tinh chỉnh sau khi có đủ dữ liệu thực tế để đánh giá.

    tools = set(qu_result.get("tool_selection", []))

    if "VisualSearch" in tools:
        weights["visual"] = base_weight_visual
    if "TextSearch" in tools:
        weights["text"] = base_weight_text
    if "ObjectSearch" in tools:
        weights["object"] = base_weight_object
    if "AudioSearch" in tools:
        weights["audio"] = base_weight_audio

    # Normalize weights so they sum to 1, or handle as desired.
    # For RRF, absolute weights don't strictly need to sum to 1,
    # but a relative scaling can be useful. Let's make sure at least one is > 0.
    total_weight = sum(weights.values())
    if total_weight > 0:
        for k in weights:
            weights[k] /= total_weight
    else: # Fallback if no tools selected or all weights are zero
        weights["visual"] = 1.0 # Default to visual if nothing else specified

    return weights

---
## Phần 8 — Video-level Grouping (Diversification + Scene coverage + Neighbor expansion)

**Giải thích:** giờ có đủ 3 kỹ thuật đã thiết kế:
- **Neighbor expansion**: với top-20 candidate mạnh nhất, thêm các frame lân cận (±5) làm candidate bổ sung — tăng khả năng rơi đúng khoảng `[s,e]` dung sai.
- **Scene coverage**: trong mỗi video, gom các candidate gần nhau về `frame_id` (cùng 1 "cảnh") thành 1 nhóm, chỉ giữ đại diện tốt nhất mỗi nhóm — tránh 1 cảnh chiếm hết slot của video đó.
- **Diversification** (như cũ): giới hạn số candidate tối đa lấy từ mỗi video.

In [ ]:
def get_local_frame_idx_from_frame_id(video_id: str, frame_id: int):
    """Tra NGƯỢC: từ frame_id thật, tìm local_frame_idx (cột 'n') GẦN NHẤT trong
    map_keyframes_df. Dùng để XÁC THỰC candidate từ Neighbor Expansion (vốn chỉ
    đoán frame_id bằng phép cộng/trừ, chưa từng biết local_frame_idx thật) —
    nếu không tra ngược được, candidate đó sẽ "vô hình" (không có ảnh để kiểm
    tra/hiển thị), đúng vấn đề vừa gặp với L24_V032."""
    rows = map_keyframes_df[map_keyframes_df["video_id"] == video_id]
    if len(rows) == 0:
        return None
    closest_pos = (rows["frame_idx"] - frame_id).abs().idxmin()
    return int(rows.loc[closest_pos, "n"])


def add_neighbor_expansion(df: pd.DataFrame, offsets: list = [-5, 5]) -> pd.DataFrame:
    """Neighbor expansion — với mỗi candidate, thêm các frame lân cận (frame_id +- offset)
    làm candidate bổ sung, điểm thấp hơn 1 chút so với candidate gốc.

    ĐÃ SỬA 2 lỗi dữ liệu thật:
    1. frame_id có thể ÂM nếu candidate gốc có frame_id nhỏ (VD 3 - 5 = -2) —
       giờ CHẶN về >= 0 bằng clip_frame_number() (đồng thời chặn cả cận trên).
    2. local_frame_idx trước đây LUÔN None (không xác thực được, không tra được
       ảnh) — giờ TRA NGƯỢC qua map_keyframes_df để có local_frame_idx thật,
       làm candidate này XÁC THỰC được như mọi candidate khác."""
    extra_rows = []
    for _, row in df.iterrows():
        if row["frame_id"] is None:
            continue
        for offset in offsets:
            raw_frame_id = row["frame_id"] + offset
            clipped_frame_id = clip_frame_number(row["video_id"], raw_frame_id)
            if clipped_frame_id is None or clipped_frame_id == row["frame_id"]:
                continue   # bỏ qua nếu bị clip trùng lại candidate gốc (VD frame_id gốc đã là 0)
            local_idx = get_local_frame_idx_from_frame_id(row["video_id"], clipped_frame_id)
            extra_rows.append({
                "video_id": row["video_id"],
                "frame_id": clipped_frame_id,
                "local_frame_idx": local_idx,   # giờ là số thật (hoặc None nếu video không có
                                                  # trong map_keyframes_df — trường hợp hiếm)
                "rrf_score": row["rrf_score"] * 0.8,
                # Đánh dấu RÕ đây là candidate TỔNG HỢP (suy ra từ lân cận), KHÔNG phải do 1
                # module tìm kiếm thật sự tìm ra -> không tính vào "đồng thuận module" khi
                # đánh giá độ tin cậy ở bước hiển thị cuối.
                "modules": ["neighbor_expansion"],
            })
    if not extra_rows:
        return df
    combined = pd.concat([df, pd.DataFrame(extra_rows)], ignore_index=True)
    return combined.sort_values("rrf_score", ascending=False).drop_duplicates(["video_id", "frame_id"]).reset_index(drop=True)


def diversify(df: pd.DataFrame, max_per_video: int = 5, budget: int = 100,
              scene_gap_frames: int = 300, enable_neighbor_expansion: bool = True) -> pd.DataFrame:
    if len(df) == 0:
        return df

    working_df = df.copy()
    if enable_neighbor_expansion:
        expanded_top = add_neighbor_expansion(working_df.head(20).copy())
        rest = working_df.iloc[20:]
        working_df = pd.concat([expanded_top, rest], ignore_index=True)
        working_df = working_df.sort_values("rrf_score", ascending=False).drop_duplicates(["video_id", "frame_id"]).reset_index(drop=True)

    # Scene coverage: gán scene_id theo khoảng cách frame_id trong cùng video
    working_df = working_df.sort_values(["video_id", "frame_id"]).reset_index(drop=True)
    scene_ids, current_scene, prev_video, prev_frame = [], 0, None, None
    for _, row in working_df.iterrows():
        if row["video_id"] != prev_video or (prev_frame is not None and row["frame_id"] - prev_frame > scene_gap_frames):
            current_scene += 1
        scene_ids.append(f'{row["video_id"]}_{current_scene}')
        prev_video, prev_frame = row["video_id"], row["frame_id"]
    working_df["scene_id"] = scene_ids

    # Mỗi scene chỉ giữ đại diện điểm cao nhất -> tránh 1 cảnh chiếm hết slot của video
    working_df = working_df.sort_values("rrf_score", ascending=False)
    scene_best = working_df.drop_duplicates("scene_id", keep="first")

    # Diversification: xoay vòng qua các video, giới hạn max_per_video
    grouped = {vid: g.reset_index(drop=True) for vid, g in scene_best.groupby("video_id")}
    diversified, round_idx = [], 0
    while len(diversified) < budget and round_idx < max_per_video:
        for vid, g in grouped.items():
            if round_idx < len(g):
                diversified.append(g.iloc[round_idx])
        round_idx += 1

    result_df = pd.DataFrame(diversified).reset_index(drop=True)
    return result_df.sort_values("rrf_score", ascending=False).head(budget).reset_index(drop=True)

### 8.1 — Frame/time metadata: boost theo vị trí tương đối trong video (đầu/giữa/cuối)

**Giải thích:** trước đây `Frame/time metadata` chưa từng dùng. Giờ nếu câu query gợi ý rõ vị trí (`temporal_position_hint`), boost điểm các candidate rơi đúng vùng đó — dùng `clip_mapping_df` để biết tổng số keyframe mỗi video, từ đó suy ra vị trí TƯƠNG ĐỐI (0.0=đầu, 1.0=cuối) của từng candidate.

In [86]:
def apply_temporal_position_filter(df: pd.DataFrame, position_hint: str) -> pd.DataFrame:
    """Frame/time metadata — boost nhẹ các candidate rơi đúng vùng thời gian gợi ý
    (đầu/giữa/cuối video), KHÔNG loại bỏ candidate không khớp (chỉ boost, không lọc
    cứng, vì gợi ý của LLM có thể không hoàn toàn chính xác)."""
    if not position_hint or len(df) == 0:
        return df

    video_frame_counts = clip_mapping_df.groupby("video_id")["local_frame_idx"].max().to_dict()

    def position_multiplier(row):
        total = video_frame_counts.get(row["video_id"])
        lidx = row.get("local_frame_idx")
        if total is None or lidx is None or pd.isna(lidx):
            return 1.0
        ratio = lidx / max(total, 1)
        if position_hint == "start" and ratio <= 0.33:
            return 1.3
        if position_hint == "middle" and 0.33 < ratio <= 0.66:
            return 1.3
        if position_hint == "end" and ratio > 0.66:
            return 1.3
        return 1.0

    df = df.copy()
    df["rrf_score"] = df.apply(lambda r: r["rrf_score"] * position_multiplier(r), axis=1)
    return df.sort_values("rrf_score", ascending=False).reset_index(drop=True)

---
## Phần 9 — Multimodal Reranker (Gemini Vision + OCR + ASR + Caption + Context)

**Giải thích:** đủ cả 5 phần đã thiết kế:
- **Visual**: ảnh thật của candidate.
- **OCR**: chữ đọc được (nếu có).
- **ASR**: lời nói tại thời điểm gần đó (quy đổi qua fps thật để tìm đúng đoạn transcript).
- **Caption**: Gemini TỰ SINH mô tả cho ảnh — gộp chung vào 1 lần gọi (không tốn thêm API call riêng).
- **Context**: 1 frame LÂN CẬN (frame trước đó trong cùng video) được gửi kèm, giúp model kiểm tra tính nhất quán của diễn biến.

In [ ]:
_keyframe_dir_cache = {}   # cache video_id -> Path thư mục keyframe thật, tránh rglob lặp lại

def find_keyframe_dir(video_id: str, debug: bool = False):
    """Tìm thư mục keyframe của video_id — ĐỆ QUY (rglob), xử lý đúng cả trường hợp
    có thêm 1 cấp thư mục lồng (VD 'keyframes/L29/L29_V002/')."""
    if video_id in _keyframe_dir_cache:
        return _keyframe_dir_cache[video_id]
    matches = [p for p in Path(EXTRACTED_ROOT, "keyframes").rglob(video_id) if p.is_dir()]
    result = matches[0] if matches else None
    _keyframe_dir_cache[video_id] = result
    if debug:
        print(f"  [DEBUG find_keyframe_dir] video_id={video_id}")
        print(f"    Tìm dưới: {Path(EXTRACTED_ROOT, 'keyframes')}")
        print(f"    Số thư mục khớp tên '{video_id}': {len(matches)}")
        print(f"    Kết quả: {result}")
    return result


def get_keyframe_image_path(video_id: str, local_frame_idx, debug: bool = False,
                             neighbor_fallback: bool = True, max_neighbor_gap: int = 3) -> str:
    """Tìm đường dẫn ảnh keyframe thật trên Drive, dựa vào local_frame_idx (thứ tự
    keyframe trong video). Đặt debug=True để in chi tiết quá trình tìm — dùng khi
    nghi ngờ không tìm thấy ảnh (như đã gặp ở lần chạy thử đầu tiên).

    CẢI TIẾN: khi idx nằm NGOÀI phạm vi file thật có trong thư mục (VD map_keyframes_df
    khai báo video có 52 keyframe nhưng thư mục chỉ có 50 file — dấu hiệu keyframe bị thiếu
    do ghi Drive gián đoạn, cùng lớp lỗi đã gặp ở L26_V296-298), TRƯỚC ĐÂY trả về None thẳng
    ("Không tìm thấy ảnh") — hệ quả: rerank_one() cho candidate đó 0 điểm NGAY LẬP TỨC mà
    KHÔNG hỏi Gemini (xem rerank_one(), nhánh image_path is None), tức là 1 lỗi lookup dữ
    liệu bị hiểu nhầm thành "nội dung không khớp". Vì pandas.sort_values là stable sort, các
    candidate hòa 0 điểm với nhau (dù vì lý do khác nhau) có thể giữ nguyên thứ tự RRF gốc
    thay vì phản ánh mức độ liên quan thật — đây là 1 phần lý do candidate lỗi ảnh vẫn có thể
    trồi lên hạng cao. Giờ: nếu lệch không quá max_neighbor_gap so với file thật gần nhất
    (kẹp về đầu/cuối danh sách), dùng file đó thay vì bỏ hẳn -> vừa hết "Không tìm thấy ảnh",
    vừa cho candidate cơ hội được Gemini đánh giá công bằng thay vì bị loại oan.
    Đặt neighbor_fallback=False để quay lại hành vi cũ (dùng khi cần biết đúng khoảng trống
    thật — xem audit_keyframe_completeness())."""
    if local_frame_idx is None or pd.isna(local_frame_idx):
        if debug:
            print(f"  [DEBUG] local_frame_idx rỗng (None/NaN) cho video_id={video_id}")
        return None

    video_dir = find_keyframe_dir(video_id, debug=debug)
    if video_dir is None:
        if debug:
            print(f"  [DEBUG] KHÔNG tìm thấy thư mục keyframe nào cho video_id={video_id} "
                  f"dưới {Path(EXTRACTED_ROOT, 'keyframes')} -> kiểm tra lại tên thư mục thật trên Drive")
        return None

    idx = int(local_frame_idx)
    idx_plus1 = idx + 1   # phòng trường hợp file đánh số bắt đầu từ 1, không phải 0
                            # (đã xác nhận thực tế: idx=0 -> file thật tên "001.jpg")

    # SỬA LỖI (phát hiện qua test với dữ liệu giả lập khi thêm neighbor_fallback ở trên):
    # thứ tự pattern CŨ thử "{idx}.jpg" TRƯỚC "{idx_plus1}.jpg". Vì file đánh số từ 1 (đã xác
    # nhận ở trên: idx=0 -> "001.jpg"), với BẤT KỲ idx >= 1 nào, file "{idx}.jpg" CŨNG LÀ 1 FILE
    # THẬT tồn tại (chỉ là nó đại diện cho keyframe TRƯỚC ĐÓ — n=idx, tức local_frame_idx=idx-1
    # — không phải keyframe đang cần là n=idx+1). Do thử idx TRƯỚC idx_plus1, hàm CŨ trả về
    # đúng file (tồn tại) nhưng SAI KEYFRAME (lệch 1 vị trí) cho MỌI idx >= 1, chỉ tình cờ ĐÚNG
    # ở idx=0 (trường hợp duy nhất từng được xác nhận thủ công) vì "0.jpg"/"000.jpg" không tồn
    # tại nên mới rơi xuống đúng pattern idx_plus1. Ảnh hiển thị debug và ảnh Gemini dùng để
    # rerank/trả lời QA vì vậy có thể đã bị lệch 1 keyframe so với frame_id thật sự được nộp
    # (frame_id nộp bài KHÔNG bị ảnh hưởng vì đến từ get_real_frame_id(), một pipeline tra cứu
    # khác, độc lập, đã đúng từ trước — chỉ ẢNH minh hoạ/dùng để chấm điểm bởi Gemini bị sai).
    # Đảo thứ tự: thử idx_plus1 (quy ước đã xác nhận đúng) TRƯỚC, idx làm phương án dự phòng
    # (cho trường hợp hiếm gặp dataset khác đánh số từ 0).
    tried_patterns = [
        f"{idx_plus1:04d}.jpg", f"{idx_plus1:04d}.png",
        f"{idx_plus1:03d}.jpg", f"{idx_plus1:03d}.png",
        f"{idx_plus1:05d}.jpg", f"{idx_plus1:05d}.png",
        f"{idx_plus1}.jpg", f"{idx_plus1}.png",
        f"{idx:04d}.jpg", f"{idx:04d}.png", f"{idx}.jpg", f"{idx}.png",
        f"{idx:05d}.jpg", f"{idx:05d}.png", f"{idx:03d}.jpg", f"{idx:03d}.png",
    ]
    for pattern in tried_patterns:
        candidate = video_dir / pattern
        if candidate.exists():
            if debug:
                print(f"  [DEBUG] Khớp đúng: {candidate}")
            return str(candidate)

    all_files = sorted(video_dir.glob("*.jpg")) + sorted(video_dir.glob("*.png")) + sorted(video_dir.glob("*.webp"))
    if 0 <= idx < len(all_files):
        if debug:
            print(f"  [DEBUG] Không khớp tên chính xác, dùng fallback theo vị trí: {all_files[idx]}")
        return str(all_files[idx])

    if neighbor_fallback and len(all_files) > 0:
        clamped_idx = max(0, min(idx, len(all_files) - 1))
        gap = abs(idx - clamped_idx)
        if gap <= max_neighbor_gap:
            if debug:
                print(f"  [DEBUG] idx={idx} vượt phạm vi thật (chỉ có {len(all_files)} file) — "
                      f"dùng file GẦN NHẤT (lệch {gap} vị trí): {all_files[clamped_idx]}")
            return str(all_files[clamped_idx])

    if debug:
        sample = list(video_dir.iterdir())[:5]
        print(f"  [DEBUG] KHÔNG tìm thấy file nào khớp. Thư mục {video_dir} có {len(list(video_dir.iterdir()))} mục.")
        print(f"  [DEBUG] 5 file mẫu thực tế trong thư mục: {[f.name for f in sample]}")
        print(f"  [DEBUG] Đã thử các pattern: {tried_patterns}")
    return None


def audit_keyframe_completeness(video_id: str) -> dict:
    """CÔNG CỤ CHẨN ĐOÁN (mới) — chạy audit_keyframe_completeness("L21_V024") để xác định
    NGUYÊN NHÂN THẬT của lỗi "Không tìm thấy ảnh": (1) THIẾU FILE THẬT trên Drive (do ghi
    gián đoạn — cùng lớp lỗi đã gặp ở L26_V296-298, chỉ có thể fix bằng re-extract/re-upload,
    KHÔNG có cách nào fix bằng code), hay (2) chỉ là sai quy ước đặt tên/offset (giờ đã tự xử
    lý được qua get_keyframe_image_path()). So sánh số lượng map_keyframes_df khai báo với số
    file thật + rà từng vị trí (KHÔNG dùng neighbor_fallback, để thấy đúng khoảng trống)."""
    expected_rows = map_keyframes_df[map_keyframes_df["video_id"] == video_id]
    expected_count = len(expected_rows)

    video_dir = find_keyframe_dir(video_id, debug=False)
    if video_dir is None:
        print(f"  [KHÔNG TÌM THẤY THƯ MỤC KEYFRAME cho {video_id}]")
        return {"video_id": video_id, "status": "KHÔNG TÌM THẤY THƯ MỤC KEYFRAME",
                "expected_count": expected_count, "actual_count": 0, "missing_indices": []}

    all_files = sorted(video_dir.glob("*.jpg")) + sorted(video_dir.glob("*.png")) + sorted(video_dir.glob("*.webp"))
    actual_count = len(all_files)

    # SỬA: kiểm tra TRỰC TIẾP theo đúng quy ước đã xác nhận (tên file = n, KHÔNG qua
    # get_keyframe_image_path() — hàm đó có fallback đa pattern/vị trí, tốt cho pipeline chính
    # ("có ảnh còn hơn không") nhưng làm AUDIT mất độ chính xác: có thể âm thầm khớp nhầm sang
    # 1 file THẬT nhưng SAI vị trí (VD thiếu file n=51 nhưng khớp nhầm file n=50 qua pattern dự
    # phòng) thay vì báo đúng là đang thiếu. Audit cần biết đúng khoảng trống thật.
    missing_indices = []
    if "n" in expected_rows.columns:
        for n_val in sorted(expected_rows["n"].tolist()):
            n_val = int(n_val)
            exact_patterns = [f"{n_val:04d}.jpg", f"{n_val:04d}.png", f"{n_val:03d}.jpg",
                               f"{n_val:03d}.png", f"{n_val:05d}.jpg", f"{n_val:05d}.png",
                               f"{n_val}.jpg", f"{n_val}.png"]
            if not any((video_dir / p).exists() for p in exact_patterns):
                missing_indices.append(n_val - 1)   # lưu dạng local_frame_idx (0-based)

    if actual_count < expected_count:
        status = "THIẾU FILE THẬT trên Drive (cần re-extract, KHÔNG fix được bằng code)"
    elif missing_indices:
        status = "Đủ số lượng file nhưng CÓ VỊ TRÍ TRA KHÔNG RA ẢNH (khả năng lệch tên/thứ tự)"
    else:
        status = "OK — đủ số lượng, không thiếu file"

    result = {
        "video_id": video_id,
        "status": status,
        "expected_count_map_keyframes_df": expected_count,
        "actual_count_file_that": actual_count,
        "missing_indices_khong_dung_neighbor_fallback": missing_indices[:20],
        "keyframe_dir": str(video_dir),
    }
    for k, v in result.items():
        print(f"  {k}: {v}")
    return result


def get_asr_text_near_frame(video_id: str, frame_id, tolerance_sec: float = 2.0) -> str:
    """Tìm đoạn ASR transcript GẦN 1 frame cụ thể — dùng frame_id THẬT quy đổi qua
    fps (đọc trực tiếp từ map_keyframes_df)."""
    if frame_id is None or len(text_index_df) == 0:
        return None
    fps = get_video_fps(video_id)
    if fps <= 0:
        return None
    approx_time = frame_id / fps

    asr_rows = text_index_df[
        (text_index_df["video_id"] == video_id) & (text_index_df.get("source") == "ASR")
    ]
    if len(asr_rows) == 0:
        return None
    nearby = asr_rows[
        (asr_rows["start"] <= approx_time + tolerance_sec) &
        (asr_rows["end"] >= approx_time - tolerance_sec)
    ]
    return " ".join(nearby["text"].tolist()) if len(nearby) > 0 else None


def get_recap_context(video_id: str, frame_id) -> str:
    """Tìm ngữ cảnh ReCap (nếu có) — caption có trí nhớ xuyên suốt video (Offline Phần 10),
    ứng với khoảng shot CHỨA frame_id này (start_frame <= frame_id <= end_frame). Trả về
    None nếu video này không có RECAP (VD chưa kịp chạy hết 873 video) hoặc frame nằm ngoài
    mọi shot đã ghi nhận."""
    if frame_id is None or len(text_index_df) == 0 or "start_frame" not in text_index_df.columns:
        return None
    recap_rows = text_index_df[
        (text_index_df["video_id"] == video_id) & (text_index_df.get("source") == "RECAP") &
        (text_index_df["start_frame"] <= frame_id) & (text_index_df["end_frame"] >= frame_id)
    ]
    if len(recap_rows) == 0:
        return None
    return " ".join(recap_rows["text"].dropna().tolist())


def build_evidence(video_id: str, frame_id, local_frame_idx) -> dict:
    """Gom bằng chứng cho 1 candidate — ảnh thật + OCR + ASR + 1 frame CONTEXT lân cận.

    SỬA LỖI: trước đây kiểm tra `local_frame_idx is not None` — nhưng candidate đến
    từ Neighbor Expansion (Phần 8) có local_frame_idx=None, và sau khi Pandas gộp
    cột (pd.concat) với các candidate khác có số nguyên, None thường bị TỰ ĐỘNG
    CHUYỂN thành NaN (float) — mà `NaN is not None` vẫn là True, nên điều kiện lọt
    qua rồi int(NaN) mới crash. Dùng pd.notna() để bắt được CẢ None LẪN NaN."""
    has_valid_local_idx = pd.notna(local_frame_idx)   # bắt được cả None và NaN

    image_path = get_keyframe_image_path(video_id, local_frame_idx)

    ocr_text = None
    if len(text_index_df) > 0 and has_valid_local_idx:
        ocr_rows = text_index_df[
            (text_index_df["video_id"] == video_id) &
            (text_index_df.get("source") == "OCR") &
            (text_index_df.get("frame_index").astype(str) == str(int(local_frame_idx)))
        ]
        if len(ocr_rows) > 0:
            ocr_text = " ".join(ocr_rows["text"].tolist())

    asr_text = get_asr_text_near_frame(video_id, frame_id)

    context_image_path = None
    if has_valid_local_idx and int(local_frame_idx) > 0:
        context_image_path = get_keyframe_image_path(video_id, int(local_frame_idx) - 1)

    recap_context = get_recap_context(video_id, frame_id)   # THÊM: ngữ cảnh câu chuyện xuyên
                                                               # suốt video, không phụ thuộc
                                                               # riêng frame này có gì hay không

    return {"image_path": image_path, "ocr_text": ocr_text, "asr_text": asr_text,
            "context_image_path": context_image_path, "recap_context": recap_context}

In [ ]:
def rerank_one(query_text: str, evidence: dict, client=None, model: str = None) -> dict:
    """client/model=None (mặc định): dùng gemini_client_1/GEMINI_MODEL (Flash-Lite) —
    giữ nguyên hành vi cũ cho phần lớn candidate. multimodal_rerank() truyền riêng
    gemini_client_2/GEMINI_MODEL_JUDGE (Flash) để escalate CHỈ candidate hạng 1 sau
    khi đã xếp hạng thô — xem giải thích đầy đủ ở multimodal_rerank()."""
    if evidence["image_path"] is None or not os.path.exists(evidence["image_path"]):
        return {"score": 0.0, "caption": None, "reason": "Không tìm thấy ảnh keyframe"}

    with open(evidence["image_path"], "rb") as f:
        image_bytes = f.read()
    parts = [types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg")]

    context_note = "(không có ảnh context)"
    if evidence.get("context_image_path") and os.path.exists(evidence["context_image_path"]):
        with open(evidence["context_image_path"], "rb") as f:
            parts.append(types.Part.from_bytes(data=f.read(), mime_type="image/jpeg"))
        context_note = "Ảnh THỨ 2 là khung hình NGAY TRƯỚC ĐÓ trong cùng video (context)."

    recap_line = ""
    if evidence.get("recap_context"):
        recap_line = f"\nBối cảnh câu chuyện xuyên suốt video tại đây (ReCap): {evidence['recap_context']}\n"

    prompt = f"""Câu truy vấn: "{query_text}"
Ảnh 1 là candidate chính. {context_note}
OCR: {evidence['ocr_text'] or 'không có'}, ASR: {evidence.get('asr_text') or 'không có'}
{recap_line}
QUAN TRỌNG — về trường "caption": mô tả LẠI những gì BẠN THẬT SỰ THẤY trong ảnh, BẰNG
LỜI CỦA CHÍNH BẠN — TUYỆT ĐỐI KHÔNG chép lại/diễn giải lại nguyên văn câu truy vấn ở trên,
kể cả khi bạn tin ảnh khớp hoàn hảo. Nếu caption bạn viết ra giống hệt hoặc gần giống hệt
câu truy vấn, đó là dấu hiệu bạn đang ĐOÁN theo câu hỏi thay vì THẬT SỰ mô tả ảnh — hãy nhìn
kỹ lại ảnh và mô tả đúng những chi tiết cụ thể quan sát được (số người, màu sắc, tư thế...).

QUAN TRỌNG (chống bịa mô tả — phát hiện qua test thật: model từng viết caption "đĩa tôm 5
con xếp vòng cung" cho 1 ảnh THẬT SỰ chỉ có 2 người dẫn chương trình đứng trong bếp, không
hề có đĩa tôm nào): TRƯỚC KHI trả lời, hãy tự kiểm tra lại — caption bạn sắp viết có THẬT SỰ
mô tả đúng những gì đang HIỂN THỊ trong Ảnh 1 hay không, hay bạn đang vô tình mô tả 1 cảnh
tưởng tượng/hợp lý với câu hỏi nhưng KHÔNG có trong ảnh thật. Nếu không chắc chắn 100% điều
gì đó có xuất hiện trong ảnh, ĐỪNG đưa nó vào caption.

Trả về JSON: {{"caption": "<mô tả CỦA BẠN, ĐÃ tự kiểm tra khớp đúng ảnh thật>", "score": <0-1>, "reason": "<giải thích>"}}"""
    parts.append(prompt)

    call_client = client or gemini_client_1
    call_model = model or GEMINI_MODEL
    limiter = gemini_judge_rate_limiter if call_client is gemini_client_2 else gemini_rate_limiter

    # FIX (P0 — quan trọng): trước đây bắt Exception rồi trả score=0.0 NGAY LẬP TỨC —
    # nghĩa là 1 lần dính 429 (rate limit) khiến candidate ĐÚNG bị chấm "không khớp nội
    # dung" hoàn toàn oan. Giờ dùng call_gemini_with_retry() (Phần 2.1): retry có backoff
    # nếu là lỗi rate limit/quá tải, CHỈ trả score=0.0 khi đã thử hết số lần retry mà vẫn
    # lỗi (lúc đó mới hợp lý coi là "không đánh giá được").
    def _call():
        response = call_client.models.generate_content(
            model=call_model, contents=parts,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                http_options=types.HttpOptions(timeout=GEMINI_TIMEOUT_MS),
            ),
        )
        return json.loads(response.text)

    try:
        return call_gemini_with_retry(_call, limiter=limiter)
    except Exception as e:
        return {"score": 0.0, "caption": None, "reason": f"Lỗi Gemini sau nhiều lần thử: {e}"}

def multimodal_rerank(candidates_df: pd.DataFrame, query_text: str, top_n_to_rerank: int = 20) -> pd.DataFrame:
    """CẢI TIẾN (theo đề xuất): candidate hạng 1 (cái THẬT SỰ sẽ được nộp bài) được chấm
    lại LẦN 2 bằng gemini-3.5-flash (mạnh hơn Flash-Lite, đặc biệt cho phân biệt thị giác
    tinh) — thay vì nâng hẳn CẢ 20 candidate lên Flash (tốn quota/thời gian hơn nhiều lần,
    vì Flash chỉ có 10 RPM free tier so với 15 RPM của Flash-Lite — xem thảo luận rate
    limit). Đây là chiến lược coarse-to-fine, cùng tinh thần với align_event_to_frame()
    (chọn thô rồi tinh chỉnh) — chỉ tốn thêm ĐÚNG 1 lần gọi Gemini/query, không phụ thuộc
    top_n_to_rerank lớn cỡ nào."""
    if len(candidates_df) == 0:
        return candidates_df

    to_rerank = candidates_df.head(top_n_to_rerank).copy()
    remaining = candidates_df.iloc[top_n_to_rerank:].copy()

    rerank_scores, captions = [], []
    for _, row in to_rerank.iterrows():
        evidence = build_evidence(row["video_id"], row["frame_id"], row.get("local_frame_idx"))
        result = rerank_one(query_text, evidence)   # Flash-Lite, như cũ, cho TẤT CẢ candidate
        rerank_scores.append(result.get("score", 0.0))
        captions.append(result.get("caption"))

    to_rerank["rerank_score"] = rerank_scores
    to_rerank["caption"] = captions

    if len(remaining) > 0:
        remaining["rerank_score"] = 0.0
        remaining["caption"] = None

    combined = pd.concat([to_rerank, remaining], ignore_index=True)
    combined = combined.sort_values(["rerank_score", "rrf_score"], ascending=False).reset_index(drop=True)

    # ---- Escalate: chấm LẠI đúng candidate đang đứng hạng 1 bằng Flash ----
    if len(combined) > 0 and combined.iloc[0]["rerank_score"] > 0:
        top_row = combined.iloc[0]
        evidence = build_evidence(top_row["video_id"], top_row["frame_id"], top_row.get("local_frame_idx"))
        flash_result = rerank_one(query_text, evidence, client=gemini_client_2, model=GEMINI_MODEL_JUDGE)
        combined.loc[0, "rerank_score"] = flash_result.get("score", top_row["rerank_score"])
        if flash_result.get("caption"):
            combined.loc[0, "caption"] = flash_result["caption"]
        # Flash có thể KHÔNG đồng ý với Flash-Lite -> sort lại để phản ánh đúng, tránh giữ
        # nguyên thứ tự cũ nếu điểm hạng 1 giờ thấp hơn hạng 2.
        combined = combined.sort_values(["rerank_score", "rrf_score"], ascending=False).reset_index(drop=True)

    return combined


---
## Phần 10 — Temporal Reasoning (tổng quát TRAKE + QA-temporal) và LVLM Reasoning (QA)

**Thay đổi quan trọng — đúng vị trí thiết kế gốc:** `temporal_reasoning()` giờ là hàm DÙNG CHUNG cho cả TRAKE và QA có `needs_temporal=True`, và được gọi TRONG `run_pipeline()` NGAY SAU Video-level Grouping — TRƯỚC Multimodal Reranker (Phần 9), đúng như sơ đồ gốc: `TEMPORAL NEEDED? -> [CÓ] -> Temporal Reasoning -> Multimodal Reranker`.

**Vì sao vẫn an toàn về chi phí dù chạy sớm hơn:** `temporal_reasoning()` chỉ chạy cho ĐÚNG 1 video (hạng 1 sau Diversification/Frame-time filter) — không phải chạy cho nhiều candidate chưa chắc đúng. Reranker (chạy SAU) vẫn có tác dụng: xác nhận/tinh chỉnh lại các frame Temporal Reasoning vừa tìm ra, cùng các candidate khác.

**Nhánh QA (LVLM Reasoning):**
- `needs_temporal=False`: 1 ảnh — `lvlm_answer_question()`.
- `needs_temporal=True`: dùng ĐÚNG các frame đã được `temporal_reasoning()` xác định — `lvlm_answer_question_multi_frame()`.

In [ ]:
def lvlm_answer_question(video_id: str, frame_id, local_frame_idx, question: str,
                          final_candidates: pd.DataFrame = None,
                          client=None, model: str = None) -> str:
    """LVLM Reasoning cho QA (needs_temporal=False). Với câu hỏi ĐẾM SỐ LƯỢNG
    (chứa "bao nhiêu"/"mấy"/"số lượng"), dùng NHIỀU ảnh cùng video thay vì chỉ 1 —
    vì đếm số lượng thường cần quan sát CẢ QUÁ TRÌNH (VD: đếm số loại gia vị cho
    vào), không thể biết chỉ từ 1 khoảnh khắc dừng lại giữa chừng."""
    is_counting_question = any(w in question.lower() for w in ["bao nhiêu", "mấy", "số lượng"])

    parts = []
    if is_counting_question:
        # CẢI TIẾN: lấy mẫu ĐỀU trên TOÀN BỘ video (không chỉ từ candidate đã tìm được
        # qua Visual/Object Search — vốn không được chọn để "phủ đều thời gian", có thể
        # bỏ lỡ hầu hết khoảnh khắc thêm gia vị). Dùng clip_mapping_df để biết TẤT CẢ
        # keyframe của video này, rồi lấy mẫu rải đều từ đầu đến cuối.
        video_frames = clip_mapping_df[clip_mapping_df["video_id"] == video_id].sort_values("local_frame_idx")
        n_total = len(video_frames)
        if n_total > 0:
            n_sample = min(8, n_total)
            sample_positions = np.linspace(0, n_total - 1, n_sample, dtype=int)
            sampled_rows = video_frames.iloc[sample_positions]
            for _, row in sampled_rows.iterrows():
                path = get_keyframe_image_path(video_id, row["local_frame_idx"])
                if path and os.path.exists(path):
                    with open(path, "rb") as f:
                        parts.append(types.Part.from_bytes(data=f.read(), mime_type="image/jpeg"))
        context_note = (f"Đây là {len(parts)} khung hình lấy mẫu RẢI ĐỀU trên TOÀN BỘ video "
                         f"(từ đầu đến cuối, không chỉ 1 đoạn) — giúp bạn quan sát DIỄN BIẾN CẢ "
                         f"QUÁ TRÌNH để đếm chính xác hơn, thay vì chỉ nhìn 1-2 khoảnh khắc rời rạc.")
    else:
        image_path = get_keyframe_image_path(video_id, local_frame_idx)
        if image_path is None or not os.path.exists(image_path):
            return None
        with open(image_path, "rb") as f:
            parts.append(types.Part.from_bytes(data=f.read(), mime_type="image/jpeg"))

        # CẢI TIẾN (P1.3): đính kèm OCR/ASR gần frame này làm THÔNG TIN THAM KHẢO — nhiều câu
        # hỏi (địa danh, câu thơ, tên món ăn...) phụ thuộc CHỮ HIỂN THỊ TRÊN MÀN HÌNH mà Gemini
        # Vision đọc trực tiếp từ ảnh nén dễ sai (đặc biệt tiếng Việt có dấu). build_evidence()
        # đã tính sẵn 2 trường này cho bước rerank — tái dùng ở đây cho bước trả lời cuối.
        ocr_hint = None
        if len(text_index_df) > 0 and pd.notna(local_frame_idx):
            ocr_rows = text_index_df[
                (text_index_df["video_id"] == video_id) &
                (text_index_df.get("source") == "OCR") &
                (text_index_df.get("frame_index").astype(str) == str(int(local_frame_idx)))
            ]
            if len(ocr_rows) > 0:
                ocr_hint = " ".join(ocr_rows["text"].tolist())
        asr_hint = get_asr_text_near_frame(video_id, frame_id)
        recap_hint = get_recap_context(video_id, frame_id)   # THÊM: ngữ cảnh ReCap

        evidence_lines = []
        if ocr_hint:
            evidence_lines.append(f"- Chữ đọc được trên màn hình (OCR) gần khung hình này: {ocr_hint}")
        if asr_hint:
            evidence_lines.append(f"- Lời nói tại thời điểm gần đó (ASR): {asr_hint}")
        if recap_hint:
            evidence_lines.append(f"- Bối cảnh câu chuyện xuyên suốt video tại đây (ReCap): {recap_hint}")
        evidence_block = ("\n" + "\n".join(evidence_lines) + "\n") if evidence_lines else ""

        context_note = f"Đây là 1 khung hình từ video.{evidence_block}"

    if not parts:
        return None

    prompt = f"""{context_note}

Trả lời câu hỏi sau. QUAN TRỌNG — định dạng câu trả lời (P4, chưa rõ quy chế chấm QA
của BTC là exact-match hay fuzzy-match, nên ưu tiên đáp án NGẮN, DẠNG CHUẨN — rủi ro thấp
hơn hẳn 1 câu đầy đủ dài dòng ở cả 2 kiểu chấm):
- CHỈ đưa ra ĐÁP ÁN CUỐI CÙNG (1 từ, 1 cụm từ, hoặc 1 con số) — KHÔNG viết thành câu hoàn
  chỉnh, KHÔNG lặp lại câu hỏi, KHÔNG thêm "Đáp án là...", "Câu trả lời là...".
- Ví dụ ĐÚNG: "màu đỏ" (không phải "Chiếc áo có màu đỏ")
- Ví dụ ĐÚNG: "3" (không phải "Có 3 người trong ảnh")
- Giữ nguyên chính tả/dấu tiếng Việt như trong ảnh (không phiên âm/viết tắt khác đi).
- Tối đa 100 ký tự (giới hạn của BTC).

Câu hỏi: {question}

QUAN TRỌNG (chống bịa đáp án — phát hiện qua test thật: model từng tự tin trả lời 1 tên
địa danh cụ thể dù KHÔNG có bất kỳ chữ/lời nói nào trong ảnh xác nhận tên đó, chỉ đoán theo
"trông giống" 1 nơi nào đó): nếu câu hỏi yêu cầu TÊN RIÊNG (tên địa danh, tên người, tên sự
kiện, tên chương trình...), CHỈ được trả lời tên cụ thể nếu tên đó THỰC SỰ xuất hiện trong
phần "Chữ đọc được trên màn hình (OCR)" hoặc "Lời nói (ASR)" đã cung cấp ở trên. TUYỆT ĐỐI
KHÔNG suy đoán/bịa ra 1 tên nghe hợp lý chỉ vì ảnh "trông giống" nơi nào đó bạn biết — hình
ảnh trực quan KHÔNG đủ để xác định tên riêng nếu không có chữ/lời nói xác nhận.

Nếu KHÔNG đủ thông tin để xác định chính xác từ (các) hình ảnh này, trả lời ĐÚNG
câu "Không đủ thông tin để xác định" — TUYỆT ĐỐI KHÔNG trả lời mơ hồ/né tránh kiểu
"dựa vào hình ảnh..." mà không đưa ra câu trả lời cụ thể."""
    parts.append(prompt)

    # FIX (P0 — quan trọng): trước đây bắt Exception rồi trả None NGAY LẬP TỨC — 1 lần
    # dính 429 khiến câu QA đáng lẽ trả lời được lại bị bỏ trống oan. Giờ retry có backoff
    # qua call_gemini_with_retry() (Phần 2.1), chỉ trả None khi đã thử hết số lần retry.
    call_client = client or gemini_client_1
    call_model = model or GEMINI_MODEL
    limiter = gemini_judge_rate_limiter if call_client is gemini_client_2 else gemini_rate_limiter

    def _call():
        response = call_client.models.generate_content(
            model=call_model, contents=parts,
            config=types.GenerateContentConfig(http_options=types.HttpOptions(timeout=GEMINI_TIMEOUT_MS)),
        )
        return response.text.strip()

    try:
        return call_gemini_with_retry(_call, limiter=limiter)
    except Exception as e:
        print(f"  [Lỗi LVLM Reasoning sau nhiều lần thử] {e}")
        return None

# SỬA LỖI (P0.1 — BUG NGHIÊM TRỌNG): hàm lvlm_answer_question_multi_frame() được GỌI trong
# output_formatter() (Phần 12) cho nhánh QA-temporal (needs_temporal=True) nhưng TRƯỚC ĐÂY
# CHƯA TỪNG được định nghĩa ở đâu cả (chỉ có comment nói "đã chuyển qua Phần 10" nhưng thực
# tế không có) -> gây NameError, crash toàn bộ pipeline cho bất kỳ câu QA nào cần temporal
# reasoning. Định nghĩa THẬT ở đây:
def lvlm_answer_question_multi_frame(video_id: str, frame_ids: list, question: str) -> str:
    """QA-temporal (needs_temporal=True): dùng ĐÚNG các frame_id đã được temporal_reasoning()
    xác định — đây là frame THẬT từ Dense Frame Access (không phải keyframe có sẵn), nên phải
    trích bằng extract_single_frame() (giống cách Phần 14 hiển thị TRAKE), KHÔNG dùng
    get_keyframe_image_path(). Gửi TẤT CẢ ảnh cùng lúc cho Gemini Vision để trả lời câu hỏi có
    tính chuỗi sự kiện/thời gian — cùng tinh thần với nhánh đếm số lượng ở trên."""
    valid_frame_ids = [f for f in frame_ids if f is not None]
    if not valid_frame_ids:
        return None

    video_path = find_video_file(video_id)
    if video_path is None:
        return None

    parts = []
    tmp_paths = []
    try:
        for i, frame_id in enumerate(valid_frame_ids):
            tmp_path = f"/content/qa_temporal_frame_{i}.jpg"
            extract_single_frame(video_path, frame_id, tmp_path)
            if os.path.exists(tmp_path):
                with open(tmp_path, "rb") as f:
                    parts.append(types.Part.from_bytes(data=f.read(), mime_type="image/jpeg"))
                tmp_paths.append(tmp_path)

        if not parts:
            return None

        # Gộp ASR quanh các khoảnh khắc này làm thông tin tham khảo (cùng lý do P1.3). Không
        # lấy OCR ở đây vì các frame này không có local_frame_idx tương ứng để tra cột
        # frame_index của bảng OCR.
        asr_hints = []
        for frame_id in valid_frame_ids:
            hint = get_asr_text_near_frame(video_id, frame_id)
            if hint and hint not in asr_hints:
                asr_hints.append(hint)
        asr_block = (f"\n- Lời nói (ASR) quanh các khoảnh khắc này: {' | '.join(asr_hints)}\n"
                     if asr_hints else "")

        context_note = (f"Đây là {len(parts)} khung hình liên tiếp theo ĐÚNG thứ tự thời gian, "
                         f"tương ứng với các khoảnh khắc chính trong 1 chuỗi sự kiện của video — "
                         f"dùng để hiểu bối cảnh/diễn biến trước khi trả lời.{asr_block}")

        prompt = f"""{context_note}

Trả lời câu hỏi sau. QUAN TRỌNG — định dạng câu trả lời (P4, chưa rõ quy chế chấm QA
của BTC là exact-match hay fuzzy-match, nên ưu tiên đáp án NGẮN, DẠNG CHUẨN — rủi ro thấp
hơn hẳn 1 câu đầy đủ dài dòng ở cả 2 kiểu chấm):
- CHỈ đưa ra ĐÁP ÁN CUỐI CÙNG (1 từ, 1 cụm từ, hoặc 1 con số) — KHÔNG viết thành câu hoàn
  chỉnh, KHÔNG lặp lại câu hỏi, KHÔNG thêm "Đáp án là...", "Câu trả lời là...".
- Ví dụ ĐÚNG: "màu đỏ" (không phải "Chiếc áo có màu đỏ")
- Ví dụ ĐÚNG: "3" (không phải "Có 3 người trong ảnh")
- Giữ nguyên chính tả/dấu tiếng Việt như trong ảnh (không phiên âm/viết tắt khác đi).
- Tối đa 100 ký tự (giới hạn của BTC).

Câu hỏi: {question}

QUAN TRỌNG (chống bịa đáp án — phát hiện qua test thật: model từng tự tin trả lời 1 tên
địa danh cụ thể dù KHÔNG có bất kỳ chữ/lời nói nào trong ảnh xác nhận tên đó, chỉ đoán theo
"trông giống" 1 nơi nào đó): nếu câu hỏi yêu cầu TÊN RIÊNG (tên địa danh, tên người, tên sự
kiện, tên chương trình...), CHỈ được trả lời tên cụ thể nếu tên đó THỰC SỰ xuất hiện trong
phần "Chữ đọc được trên màn hình (OCR)" hoặc "Lời nói (ASR)" đã cung cấp ở trên. TUYỆT ĐỐI
KHÔNG suy đoán/bịa ra 1 tên nghe hợp lý chỉ vì ảnh "trông giống" nơi nào đó bạn biết — hình
ảnh trực quan KHÔNG đủ để xác định tên riêng nếu không có chữ/lời nói xác nhận.

Nếu KHÔNG đủ thông tin để xác định chính xác từ (các) hình ảnh này, trả lời ĐÚNG
câu "Không đủ thông tin để xác định" — TUYỆT ĐỐI KHÔNG trả lời mơ hồ/né tránh kiểu
"dựa vào hình ảnh..." mà không đưa ra câu trả lời cụ thể."""
        parts.append(prompt)

        # FIX: cùng lý do như lvlm_answer_question() ở trên — retry qua
        # call_gemini_with_retry() thay vì trả None ngay khi dính 429/503.
        # ESCALATE: hàm này CHỈ được gọi cho candidate hạng 1 (video đã chạy Temporal
        # Reasoning đầy đủ — xem output_formatter()), nên luôn dùng Flash thay vì
        # Flash-Lite, cùng chiến lược "chỉ nâng đúng chỗ sẽ nộp bài" như rerank/QA thường.
        def _call():
            response = gemini_client_2.models.generate_content(
                model=GEMINI_MODEL_JUDGE, contents=parts,
                config=types.GenerateContentConfig(http_options=types.HttpOptions(timeout=GEMINI_TIMEOUT_MS)),
            )
            return response.text.strip()
        return call_gemini_with_retry(_call, limiter=gemini_judge_rate_limiter)
    except Exception as e:
        print(f"  [Lỗi LVLM Reasoning multi-frame sau nhiều lần thử] {e}")
        return None
    finally:
        for p in tmp_paths:
            if os.path.exists(p):
                os.remove(p)

In [90]:
def trake_multiframe_formatter(temporal_result: dict, N: int) -> dict:
    """Định dạng kết quả TRAKE: video_id và danh sách N frame_ids.
    Nếu số frame tìm được ít hơn N, sẽ padding thêm None.
    Nếu nhiều hơn, sẽ lấy N frame đầu tiên.
    """
    video_id = temporal_result.get("video_id")
    found_frames = temporal_result.get("frame_ids", [])

    # Đảm bảo đủ N frame_id
    final_frames = list(found_frames)
    if len(final_frames) < N:
        final_frames.extend([None] * (N - len(final_frames)))
    else:
        final_frames = final_frames[:N]

    return {"video_id": video_id, "frame_ids": final_frames}

In [91]:
import shutil

def find_video_file(video_id: str) -> str:
    """Tìm file video thật trên Drive (đã giải nén sẵn ở extracted/video/)."""
    candidates = list(Path(EXTRACTED_ROOT, "video").rglob(f"{video_id}.mp4"))
    return str(candidates[0]) if candidates else None


# clip_frame_number() và _video_max_frame_lookup ĐÃ DỜI LÊN Phần 5 (cell sau
# get_video_fps()) — vì add_neighbor_expansion() ở Phần 8 cần dùng SỚM HƠN,
# trước khi tới Phần 10 này. Xem định nghĩa đầy đủ ở đó.


def extract_single_frame(video_path: str, frame_number: int, output_path: str):
    """Trích xuất ĐÚNG 1 frame cụ thể từ video — dùng cho hiển thị kết quả TRAKE
    (Phần 14) VÀ cho QA-temporal (dưới đây), vì cả 2 đều cần frame từ Dense decode,
    không có sẵn trong Keyframes. Đặt ở đây (Phần 10) để dùng được SỚM, tránh lỗi
    'gọi trước khi định nghĩa' nếu để ở Phần 14 (đứng sau Phần 13 nơi thực sự gọi)."""
    cmd = (
        f'ffmpeg -y -i "{video_path}" '
        f'-vf "select=\'eq(n,{frame_number})\'" -vsync vfr -frames:v 1 '
        f'"{output_path}" -loglevel quiet'
    )
    os.system(cmd)


def enforce_event_ordering(windows: list, min_gap_frames: int = 150) -> list:
    """Event ordering — ÉP BUỘC window sau LUÔN bắt đầu SAU khi window trước kết
    thúc, VÀ cách nhau ÍT NHẤT min_gap_frames (mặc định 150 frame ~ 5-6 giây ở
    25fps). Trước đây chỉ ép đúng THỨ TỰ, không ép KHOẢNG CÁCH — dẫn đến tình
    huống thật đã gặp: 3/4 event bị dồn vào 1 khoảng rất hẹp (chỉ ~130-280 frame),
    trông gần như giống hệt nhau, không phân biệt được các giai đoạn khác nhau."""
    adjusted = []
    prev_end = -1
    for window in windows:
        if window is None:
            adjusted.append(None)
            continue
        start, end = window
        min_allowed_start = prev_end + min_gap_frames if prev_end >= 0 else start
        if start < min_allowed_start:
            width = end - start
            start = min_allowed_start
            end = start + width
            print(f"  [Event ordering] Window quá gần event trước (< {min_gap_frames} frame) "
                  f"-> điều chỉnh thành ({start}, {end})")
        adjusted.append((start, end))
        prev_end = end
    return adjusted


def estimate_event_windows(video_id: str, events: list, buffer_frames: int = 75) -> list:
    """Event boundaries — với mỗi event, search CLIP CHỈ TRONG video này (không phải
    toàn kho) để tìm vị trí thô, rồi mở rộng +-buffer_frames làm window an toàn.
    Đơn vị window là SỐ FRAME thật. CUỐI CÙNG áp dụng Event ordering để đảm bảo
    window không chồng lấn/đảo ngược thứ tự."""
    video_rows = clip_mapping_df[clip_mapping_df["video_id"] == video_id]
    if len(video_rows) == 0:
        return [None] * len(events)

    windows = []
    for event in events:
        action_text = (event.get("action") or "").replace("_", " ")
        query_vector = clip_model.encode(action_text).astype("float32").reshape(1, -1)
        faiss.normalize_L2(query_vector)

        best_score, best_local_idx = -1.0, None
        for pos in video_rows.index:
            vec = clip_index.reconstruct(int(pos)).reshape(1, -1)
            sim = float(np.dot(query_vector, vec.T).item())   # .item() tránh DeprecationWarning
            if sim > best_score:
                best_score = sim
                best_local_idx = int(video_rows.loc[pos, "local_frame_idx"])

        if best_local_idx is None:
            windows.append(None)
            continue

        center_frame_id = get_real_frame_id(video_id, best_local_idx)
        if center_frame_id is None:
            windows.append(None)
            continue

        w_start = clip_frame_number(video_id, center_frame_id - buffer_frames)
        w_end = clip_frame_number(video_id, center_frame_id + buffer_frames)
        windows.append((w_start, w_end))

    return enforce_event_ordering(windows)

In [ ]:
def dense_decode_window(video_path: str, start_frame: int, end_frame: int, output_dir: str) -> list:
    """Dense Frame Access — decode MỌI frame gốc trong khoảng [start_frame, end_frame],
    dùng filter select='between(n,...)' của ffmpeg (n = số thứ tự frame, không cần biết fps)."""
    os.makedirs(output_dir, exist_ok=True)
    cmd = (
        f'ffmpeg -y -i "{video_path}" '
        f'-vf "select=\'between(n,{start_frame},{end_frame})\'" -vsync vfr '
        f'"{output_dir}/frame_%04d.jpg" -loglevel quiet'
    )
    os.system(cmd)
    return sorted(Path(output_dir).glob("frame_*.jpg"))


def align_event_to_frame(video_path: str, window: tuple, event_description: str,
                          refine: bool = True, refine_radius: int = 12) -> int:
    """ESCALATE lên gemini-3.5-flash (không phải Flash-Lite) cho CẢ 2 lượt gọi (thô + tinh
    chỉnh) — đây là việc khó nhất về thị giác trong cả hệ thống (phân biệt frame liên tiếp
    sát nhau, khoảng đáp án đúng của BTC rất hẹp <10 frame), và số lượt gọi/query RẤT ÍT
    (chỉ 1-2 lượt/event, vài event/câu TRAKE) nên chi phí tăng thêm không đáng kể so với
    lợi ích về độ chính xác.

    Sequence matching + Time alignment — decode dense trong window, gửi 1 vài frame
    đại diện (không phải TẤT CẢ, để tiết kiệm) cho Gemini Vision chọn frame khớp nhất.

    CẢI TIẾN (P1.2 — coarse-to-fine): quy chế chấm điểm TRAKE quy định khoảng đáp án đúng của
    MỖI event thường RẤT HẸP (< 10 frame). Bước chọn thô ban đầu chỉ lấy ~10 ảnh mẫu cách nhau
    ~15 frame (với window mặc định 150 frame) — dù Gemini chọn đúng khu vực, sai số có thể lên
    tới ±15 frame, dễ lọt ra ngoài khoảng hẹp đó. Nên sau bước thô, decode DÀY (mỗi frame) trong
    1 cửa sổ hẹp quanh kết quả thô (mặc định ±12 frame) và cho Gemini chọn LẦN CUỐI trong tập hẹp
    này. Tốn thêm ~1 lần gọi Gemini/event, đổi lại tăng đáng kể xác suất trúng khoảng đáp án.
    Đặt refine=False để quay lại hành vi cũ (chỉ 1 bước thô) nếu cần tiết kiệm quota."""
    if window is None:
        return None
    start_frame, end_frame = window
    tmp_dir = f"/content/trake_window_{start_frame}_{end_frame}"

    frame_files = dense_decode_window(video_path, start_frame, end_frame, tmp_dir)
    if not frame_files:
        shutil.rmtree(tmp_dir, ignore_errors=True)
        return start_frame + (end_frame - start_frame) // 2   # fallback: giữa window

    step = max(1, len(frame_files) // 10)
    sample_files = frame_files[::step][:10]   # lấy tối đa ~10 ảnh đại diện, cách quãng

    image_parts = []
    for f in sample_files:
        with open(f, "rb") as fp:
            image_parts.append(types.Part.from_bytes(data=fp.read(), mime_type="image/jpeg"))

    prompt = f"""Đây là {len(sample_files)} khung hình liên tiếp trong 1 video, đánh số theo
thứ tự thời gian (ảnh 1 là sớm nhất). Tìm khung hình khớp NHẤT với mô tả sau:

"{event_description}"

Trả về ĐÚNG 1 JSON: {{"best_index": <số thứ tự ảnh, từ 1 đến {len(sample_files)}>}}"""

    coarse_frame_number = None
    try:
        # Thêm khoảng nghỉ nhỏ để tránh spam API liên tục trong vòng lặp TRAKE
        time.sleep(1.0)
        result = call_gemini_json(image_parts + [prompt], client=gemini_client_2, model=GEMINI_MODEL_JUDGE, max_retries=5)
        best_idx = max(0, min(result.get("best_index", 1) - 1, len(sample_files) - 1))
        chosen_file = sample_files[best_idx]
        coarse_frame_number = start_frame + frame_files.index(chosen_file)
    except Exception as e:
        print(f"  [Lỗi align_event_to_frame - bước thô] {e}")
        coarse_frame_number = start_frame + (end_frame - start_frame) // 2
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    if not refine or coarse_frame_number is None:
        return coarse_frame_number

    # ---- Bước tinh chỉnh: decode dày (mỗi frame) quanh coarse_frame_number ----
    refine_start = max(start_frame, coarse_frame_number - refine_radius)
    refine_end = min(end_frame, coarse_frame_number + refine_radius)
    refine_tmp_dir = f"/content/trake_refine_{refine_start}_{refine_end}"

    try:
        refine_files = dense_decode_window(video_path, refine_start, refine_end, refine_tmp_dir)
        if not refine_files:
            return coarse_frame_number

        refine_sample = refine_files[:15]   # cửa sổ hẹp -> không cần skip, gửi tối đa 15 ảnh
        refine_parts = []
        for f in refine_sample:
            with open(f, "rb") as fp:
                refine_parts.append(types.Part.from_bytes(data=fp.read(), mime_type="image/jpeg"))

        refine_prompt = f"""Đây là {len(refine_sample)} khung hình liên tiếp SÁT NHAU (mỗi ảnh
cách nhau đúng 1 frame) trong 1 video, đánh số theo thứ tự thời gian (ảnh 1 là sớm nhất). Đây là
bước TINH CHỈNH — hãy chọn CHÍNH XÁC khung hình khớp NHẤT (không phải khu vực gần đúng) với mô tả
sau:

"{event_description}"

Trả về ĐÚNG 1 JSON: {{"best_index": <số thứ tự ảnh, từ 1 đến {len(refine_sample)}>}}"""

        time.sleep(1.0)
        refine_result = call_gemini_json(refine_parts + [refine_prompt], client=gemini_client_2, model=GEMINI_MODEL_JUDGE, max_retries=5)
        refine_idx = max(0, min(refine_result.get("best_index", 1) - 1, len(refine_sample) - 1))
        refine_chosen_file = refine_sample[refine_idx]
        return refine_start + refine_files.index(refine_chosen_file)
    except Exception as e:
        print(f"  [Lỗi align_event_to_frame - bước tinh chỉnh] {e} -> dùng kết quả thô")
        return coarse_frame_number
    finally:
        shutil.rmtree(refine_tmp_dir, ignore_errors=True)

In [93]:
def temporal_reasoning(qu_result: dict, top_video_id: str) -> dict:
    """Temporal Reasoning (TỔNG QUÁT — dùng chung cho TRAKE và QA-temporal).
    SỬA LỖI: Kiểm tra kiểu dữ liệu của temporal_relations để tránh AttributeError."""
    if qu_result["task_type"] == "TRAKE":
        raw_conditions = qu_result["scene"]["conditions"]
        normalized_conditions = []
        for i, c in enumerate(raw_conditions):
            if isinstance(c, dict):
                normalized_conditions.append({
                    "order": c.get("order", i + 1),
                    "action": c.get("action") or c.get("event") or "",
                })
            else:
                normalized_conditions.append({"order": i + 1, "action": str(c)})
        events = sorted(normalized_conditions, key=lambda e: e["order"])
    else:
        # QA-temporal hoặc Textual_KIS có needs_temporal=True
        temporal_rel = qu_result.get("temporal_relations")
        # SỬA LỖI TẠI ĐÂY: Chỉ gọi .keys() nếu là dict
        if isinstance(temporal_rel, dict):
            events = [{"order": i + 1, "action": k} for i, k in enumerate(temporal_rel.keys())]
        else:
            events = []

        if not events:
            events = [{"order": i + 1, "action": a} for i, a in enumerate(qu_result.get("actions", [])[:2])]

    if not events:
        return {"video_id": top_video_id, "frame_ids": [], "events": []}

    video_path = find_video_file(top_video_id)
    if video_path is None:
        print(f"  [LỖI] Không tìm thấy file video cho {top_video_id}")
        return {"video_id": top_video_id, "frame_ids": [None] * len(events), "events": events}

    windows = estimate_event_windows(top_video_id, events)

    frame_ids = []
    for event, window in zip(events, windows):
        action = (event.get("action") or "").replace("_", " ")
        frame_id = align_event_to_frame(video_path, window, action)
        frame_ids.append(frame_id)

    return {"video_id": top_video_id, "frame_ids": frame_ids, "events": events}

---
## Phần 11 — Output Formatter (định tuyến theo task_type)

**Thay đổi:** giờ nhận thêm `temporal_result` (kết quả `temporal_reasoning()` đã chạy TRƯỚC Reranker, ở Phần 13) làm tham số — KHÔNG tự gọi Temporal Reasoning bên trong nữa. Output Formatter giờ CHỈ làm đúng việc "định dạng", đúng vai trò ban đầu.

In [ ]:
def output_formatter(qu_result: dict, final_candidates: pd.DataFrame, temporal_result: dict = None,
                      answer_top_n: int = 20) -> dict:
    """answer_top_n: số candidate ĐẦU TIÊN được sinh answer THẬT cho QA — mặc định 20
    (bao phủ mốc chấm điểm R@20, không chỉ R@1/R@5 như bản trước). QUAN TRỌNG: nếu
    answer=None ở 1 candidate dù video_id/frame_id ĐÚNG, candidate đó vẫn bị chấm
    R-Score=0 (công thức chấm QA cần ĐỦ CẢ 3: video + frame + answer đúng) — đây là
    lý do bản cũ (chỉ top-5) làm mất điểm oan ở R@20/R@50/R@100. Tăng answer_top_n
    lên tối đa 100 nếu muốn phủ hết mọi mốc chấm, đánh đổi bằng thời gian/API
    call nhiều hơn tương ứng (mỗi candidate là ít nhất 1 lần gọi Gemini, câu hỏi
    dạng đếm số lượng còn tốn hơn — xem lvlm_answer_question())."""
    task_type = qu_result.get("task_type", "Textual_KIS")

    if task_type == "TRAKE":
        if temporal_result is None:
            return {"task_type": "TRAKE", "video_id": None, "frame_ids": []}
        N = qu_result.get("N") or len(temporal_result["frame_ids"])
        formatted = trake_multiframe_formatter(temporal_result, N)
        return {"task_type": "TRAKE", **formatted}

    elif task_type == "QA":
        # FIX BUG NGHIÊM TRỌNG (phát hiện qua test thật — câu "cân cá"): bản trước ÉP CỨNG
        # candidate hạng 1 = frame ĐẦU TIÊN mà temporal_reasoning() chọn (trước khi rerank),
        # bỏ qua HOÀN TOÀN kết quả multimodal_rerank() ngay sau đó — dù rerank_one() đã tự
        # chấm candidate đó 0 điểm ("không có cảnh khớp") và tìm ra 1 frame KHÁC ĐÚNG HƠN
        # trong CHÍNH video đó (rerank_score cao). Hệ quả: hạng 1 bị sai dù hệ thống đã biết
        # câu trả lời đúng nằm ở đâu, và cùng 1 (video_id, frame_id) bị lặp 2 lần trong kết
        # quả (1 lần "ép" làm hạng 1, 1 lần xuất hiện lại ở vị trí gốc trong final_candidates).
        #
        # SỬA: KHÔNG ép hạng nữa — tin tưởng hoàn toàn thứ tự multimodal_rerank() đã tính
        # (giống hệt nhánh QA thường). Temporal Reasoning vẫn được tận dụng, nhưng chỉ để
        # "tăng cường" câu trả lời (dùng multi-frame, nhiều ngữ cảnh hơn) cho ĐÚNG candidate
        # nào trong final_candidates thật sự thuộc video mà Temporal Reasoning đã phân tích —
        # bất kể candidate đó xếp hạng bao nhiêu, không tạo dòng mới, không trùng lặp.
        top_video_id = temporal_result.get("video_id") if temporal_result else None
        temporal_frame_ids = temporal_result.get("frame_ids", []) if temporal_result else []
        multi_frame_used = False   # chỉ tăng cường ĐÚNG 1 candidate đầu tiên khớp video

        ranked = []
        for rank, row in final_candidates.iterrows():
            answer = None
            if rank < answer_top_n:
                use_multi_frame = (
                    not multi_frame_used and qu_result.get("needs_temporal", False)
                    and temporal_result is not None and row["video_id"] == top_video_id
                )
                if use_multi_frame:
                    # Tăng cường: dùng NHIỀU frame (ngữ cảnh rộng hơn) cho candidate này —
                    # lvlm_answer_question_multi_frame() đã mặc định dùng Flash (P1).
                    answer = lvlm_answer_question_multi_frame(row["video_id"], temporal_frame_ids,
                                                                 qu_result.get("question", ""))
                    multi_frame_used = True
                elif rank == 0:
                    # ESCALATE: candidate hạng 1 THẬT (theo rerank, không phải ép) dùng Flash.
                    answer = lvlm_answer_question(row["video_id"], row["frame_id"],
                                                    row.get("local_frame_idx"), qu_result.get("question", ""),
                                                    final_candidates=final_candidates,
                                                    client=gemini_client_2, model=GEMINI_MODEL_JUDGE)
                else:
                    answer = lvlm_answer_question(row["video_id"], row["frame_id"],
                                                    row.get("local_frame_idx"), qu_result.get("question", ""),
                                                    final_candidates=final_candidates)
            ranked.append({"rank": rank + 1, "video_id": row["video_id"],
                            "frame_id": row["frame_id"], "local_frame_idx": row.get("local_frame_idx"),
                            "answer": answer, "rerank_score": row.get("rerank_score"),
                            "caption": row.get("caption"), "modules": row.get("modules")})
        return {"task_type": "QA", "ranked_results": ranked}

    else:   # Textual_KIS
        ranked = [{"rank": i + 1, "video_id": row["video_id"], "frame_id": row["frame_id"],
                   "local_frame_idx": row.get("local_frame_idx"),
                   # THÊM: giữ lại rerank_score/caption mà multimodal_rerank() đã tính (Gemini
                   # tự chấm điểm + mô tả từng candidate) — trước đây bị bỏ ở bước format cuối,
                   # khiến không có cách nào xem LẠI vì sao Gemini xếp hạng như vậy.
                   "rerank_score": row.get("rerank_score"), "caption": row.get("caption"),
                   "modules": row.get("modules")}
                  for i, row in final_candidates.iterrows()]
        return {"task_type": "Textual_KIS", "ranked_results": ranked}

---
## Phần 12 — Final Verification (đơn giản hóa)

**Giải thích:** kiểm tra nhanh trước khi coi là kết quả cuối — có candidate nào không, có duplicate không, có None lẫn vào không (dấu hiệu lỗi mapping).

In [95]:
def final_verification(output: dict) -> dict:
    task_type = output["task_type"]

    if task_type == "TRAKE":
        n_valid = sum(1 for f in output["frame_ids"] if f is not None)
        print(f"[Verify] TRAKE: video={output['video_id']}, {n_valid}/{len(output['frame_ids'])} frame hợp lệ")
    else:
        results = output["ranked_results"]
        # FIX VERIFY: Kiểm tra local_frame_idx thay vì frame_id để phát hiện lỗi thiếu ảnh
        n_missing_img = sum(1 for r in results if r.get("local_frame_idx") is None)

        seen = set()
        n_dup = 0
        for r in results:
            key = (r["video_id"], r["frame_id"])
            if key in seen: n_dup += 1
            seen.add(key)

        print(f"[Verify] {task_type}: {len(results)} dòng, {n_missing_img} local_frame_idx=None (lỗi ảnh), {n_dup} trùng lặp")
        if n_missing_img > 0 or n_dup > 0:
            print("  [CẢNH BÁO] Phát hiện candidate lỗi ảnh hoặc trùng lặp trong kết quả nộp bài!")

    return output

---
## Phần 13 — Hàm điều phối chính (`run_pipeline`)

**Giải thích:** `run_pipeline()` là hàm TRUNG TÂM — nhận 1 câu query, tự động chạy đúng nhánh dựa trên `task_type` (Gemini xác định, hoặc ép buộc qua `known_task_type` khi đã biết chắc từ tên file BTC cấp — xem Phần 15). Không có cell test riêng ở đây nữa (đã gọn lại) — dùng trực tiếp qua `process_query_package()` ở Phần 15.

In [ ]:
# normalize_tool_selection() được gọi trong run_pipeline() (hàm chạy pipeline thật,
# dùng bởi process_query_package() ở Phần 15) — định nghĩa ở đây, TRƯỚC khi được dùng.

def normalize_tool_selection(raw_tools) -> set:
    """Chuẩn hoá tool_selection từ output Gemini (Phần 6, QU_PROMPT_TEMPLATE) thành
    1 set tên tool hợp lệ, dùng cho các check kiểu 'VisualSearch' in tools.
    Xử lý các trường hợp thực tế có thể gặp:
    - None / rỗng: Gemini không trả field này hoặc trả rỗng -> fallback về {'VisualSearch'}
      để pipeline KHÔNG BAO GIỜ chạy với 0 tool nào (tránh trả về hoàn toàn rỗng cho
      1 query hợp lệ chỉ vì lỗi 1 field phụ).
    - String đơn thay vì list: bọc lại thành list.
    - Sai case/khoảng trắng thừa (VD 'visualsearch', ' VisualSearch '): so khớp
      không phân biệt hoa/thường sau khi strip().
    - Tên không khớp tool nào (Gemini hallucinate tên khác): bỏ qua phần tử đó,
      không raise lỗi.
    """
    VALID_TOOLS = {"VisualSearch", "TextSearch", "ObjectSearch", "AudioSearch"}

    if raw_tools is None:
        return {"VisualSearch"}
    if isinstance(raw_tools, str):
        raw_tools = [raw_tools]

    normalized = set()
    for t in raw_tools:
        if not isinstance(t, str):
            continue
        t_clean = t.strip()
        for valid in VALID_TOOLS:
            if t_clean.lower() == valid.lower():
                normalized.add(valid)
                break

    return normalized if normalized else {"VisualSearch"}


---
### 14.0 — P1: Double-check video ứng viên cho TRAKE khi hạng 1/2 sát điểm

**Giải thích:** TRAKE chấm all-or-nothing THEO VIDEO — chọn sai video (dù logic tìm frame bên trong tốt cỡ nào) làm mất trắng cả câu. Trước đây hệ thống tin tuyệt đối vào video hạng 1 sau RRF+Diversify, không có bước xác nhận lại. Hàm này chấm nhanh mức độ TOÀN BỘ chuỗi frame đã align khớp với TOÀN BỘ chuỗi event mô tả — dùng để so sánh 2 video ứng viên khi điểm RRF của chúng quá sát nhau để tin cậy tuyệt đối vào thứ hạng thô.


---
### 14.0b — DANTE: Dynamic Alignment of Narrative Temporal Events

**Thay thế cách làm cũ cho TRAKE** (`estimate_event_windows` + `enforce_event_ordering` + `align_event_to_frame` — tìm từng event ĐỘC LẬP bằng Gemini, dễ chọn nhầm cảnh vì mỗi event chỉ nhìn 1 từ/cụm ngắn, không biết gì về các event khác). DANTE tìm CẢ N event CÙNG LÚC bằng quy hoạch động — đảm bảo đúng thứ tự thời gian NGAY TRONG công thức toán, không cần Gemini cho phần lõi (chỉ dùng vector CLIP đã có sẵn từ Offline). Kỹ thuật lấy từ paper đội AIO_Owlgorithms (Outstanding TRAKE, AIC HCMC 2025).

**Log chi tiết bật sẵn** (`verbose=True`) — in ra: top-3 điểm CLIP riêng lẻ của từng event, kết quả nếu chọn ĐỘC LẬP (cách cũ, có thể đảo thứ tự), và kết quả DANTE thật sự chọn — để đối chiếu trực quan khi test.


In [ ]:
def dante_align_events(video_id: str, event_texts: list, lambda_penalty: float = 0.001,
                        verbose: bool = True) -> dict:
    """Tìm chuỗi N frame khớp N event, đảm bảo đúng thứ tự thời gian, bằng quy hoạch động
    thuần túy trên vector CLIP đã có sẵn (KHÔNG gọi Gemini). Công thức:
        DP[i][t] = S[i][t] + max_{τ<t}( DP[i-1][τ] − λ·(t−τ) )
    Trả về dict {video_id, score, frame_ids, local_frame_idxs, positions}.
    """
    video_rows = clip_mapping_df[clip_mapping_df["video_id"] == video_id]
    if len(video_rows) == 0:
        if verbose:
            print(f"  [DANTE] {video_id}: không có trong clip_mapping_df -> bỏ qua")
        return {"video_id": video_id, "score": -1e9,
                "frame_ids": [None] * len(event_texts), "local_frame_idxs": [None] * len(event_texts)}

    # Sắp theo local_frame_idx tăng dần -> đảm bảo "t càng lớn = càng về sau trong video"
    video_rows = video_rows.sort_values("local_frame_idx")
    faiss_positions = video_rows.index.to_numpy()          # vị trí THẬT trong FAISS index toàn cục
    local_idxs = video_rows["local_frame_idx"].to_numpy()  # để tra ngược frame_id thật sau này
    T = len(video_rows)
    N = len(event_texts)

    if verbose:
        print(f"  [DANTE] {video_id}: {T} keyframe, {N} event, λ={lambda_penalty}")

    # ---- Bước 1: Ma trận vector CLIP của TOÀN BỘ keyframe video này (T x 512) ----
    frame_vectors = np.vstack([clip_index.reconstruct(int(p)) for p in faiss_positions]).astype("float32")
    faiss.normalize_L2(frame_vectors)

    # ---- Bước 2: Encode N event, tính S[i][t] bằng 1 phép nhân ma trận (N x T) ----
    event_vectors = np.vstack([clip_model.encode(e) for e in event_texts]).astype("float32")
    faiss.normalize_L2(event_vectors)
    S = event_vectors @ frame_vectors.T

    if verbose:
        for i, e in enumerate(event_texts):
            top3 = np.argsort(-S[i])[:3]
            print(f"    E{i+1} ('{e[:60]}'): top-3 điểm riêng lẻ = "
                  f"{[(int(t), round(float(S[i, t]), 3)) for t in top3]}")

    # ---- Bước 3: Quy hoạch động (running_max — O(N*T), không quét lại từ đầu mỗi lần) ----
    DP = np.full((N, T), -1e9, dtype="float64")
    backtrack = np.full((N, T), -1, dtype="int64")
    DP[0] = S[0]

    for i in range(1, N):
        running_max, running_arg = -1e9, -1
        for t in range(T):
            if t > 0:
                candidate = DP[i - 1, t - 1] + lambda_penalty * (t - 1)
                if candidate > running_max:
                    running_max, running_arg = candidate, t - 1
            DP[i, t] = S[i, t] + running_max - lambda_penalty * t
            backtrack[i, t] = running_arg

    # ---- Bước 4: Truy vết ngược lấy chuỗi vị trí tối ưu ----
    best_t = int(np.argmax(DP[N - 1]))
    best_score = float(DP[N - 1, best_t])
    path = [best_t]
    for i in range(N - 1, 0, -1):
        best_t = int(backtrack[i, best_t])
        path.append(best_t)
    path.reverse()

    # ---- Bước 5: Quy đổi vị trí -> frame_id thật ----
    frame_ids, local_frame_idxs = [], []
    for t in path:
        lidx = int(local_idxs[t])
        frame_ids.append(get_real_frame_id(video_id, lidx))
        local_frame_idxs.append(lidx)

    if verbose:
        naive_path = [int(np.argmax(S[i])) for i in range(N)]
        flag = "  <- ĐẢO THỨ TỰ! (đúng kiểu lỗi cách làm cũ dễ gặp)" if naive_path != sorted(naive_path) else "  (tình cờ vẫn đúng thứ tự)"
        print(f"    Nếu chọn ĐỘC LẬP từng event (cách cũ): vị trí = {naive_path}{flag}")
        print(f"    DANTE chọn (đảm bảo đúng thứ tự): vị trí = {path} -> frame_id = {frame_ids}")
        print(f"    Điểm DANTE tổng: {best_score:.4f}")

    return {"video_id": video_id, "score": best_score, "frame_ids": frame_ids,
            "local_frame_idxs": local_frame_idxs, "positions": path}


def dante_select_best_video(candidate_video_ids: list, event_texts: list,
                             lambda_penalty: float = 0.001, verbose: bool = True) -> dict:
    """Chạy DANTE cho NHIỀU video ứng viên cùng lúc (rẻ — không tốn Gemini), chọn video có
    điểm DANTE cao nhất. Thay thế score_trake_video_match() (P1 cũ, chỉ so được 2 video vì
    mỗi lần so tốn thêm lượt gọi Flash) — giờ có thể xét nhiều video hơn hẳn gần như miễn phí."""
    results = []
    for vid in candidate_video_ids:
        r = dante_align_events(vid, event_texts, lambda_penalty=lambda_penalty, verbose=verbose)
        results.append(r)
        if verbose:
            print(f"  [DANTE] {vid}: điểm tổng = {r['score']:.4f}\n")

    best = max(results, key=lambda r: r["score"])
    if verbose:
        print(f"  [DANTE] === Video được chọn: {best['video_id']} (điểm {best['score']:.4f}) ===")
    return best


In [ ]:
def score_trake_video_match(video_id: str, temporal_result: dict, events: list) -> float:
    """Chấm 0-1 mức độ TOÀN BỘ chuỗi frame đã align (temporal_result) khớp với TOÀN BỘ
    chuỗi event mô tả — KHÁC với align_event_to_frame() (chấm từng event riêng lẻ để chọn
    frame). Dùng gemini-3.5-flash vì đây là quyết định ảnh hưởng CẢ CÂU (chọn sai = mất
    trắng), số lượt gọi RẤT ÍT (chỉ khi hạng 1/2 sát điểm mới trigger, tối đa 2 lần/câu)."""
    video_path = find_video_file(video_id)
    if video_path is None:
        return 0.0
    frame_ids = [f for f in temporal_result.get("frame_ids", []) if f is not None]
    if not frame_ids:
        return 0.0

    parts, tmp_paths = [], []
    try:
        for i, fid in enumerate(frame_ids):
            tmp_path = f"/content/trake_video_check_{i}.jpg"
            extract_single_frame(video_path, fid, tmp_path)
            if os.path.exists(tmp_path):
                with open(tmp_path, "rb") as f:
                    parts.append(types.Part.from_bytes(data=f.read(), mime_type="image/jpeg"))
                tmp_paths.append(tmp_path)
        if not parts:
            return 0.0

        events_desc = "\n".join(f"{i+1}. {e.get('action', '')}" for i, e in enumerate(events))
        prompt = f"""Đây là {len(parts)} khung hình liên tiếp theo thứ tự thời gian, được cho là khớp với chuỗi sự kiện sau:
{events_desc}

Đánh giá mức độ TOÀN BỘ chuỗi ảnh này khớp với TOÀN BỘ chuỗi sự kiện mô tả (0.0 = hoàn toàn không khớp, 1.0 = khớp hoàn hảo từng bước).
Trả về ĐÚNG 1 JSON: {{"score": <0-1>}}"""
        parts.append(prompt)
        result = call_gemini_json(parts, client=gemini_client_2, model=GEMINI_MODEL_JUDGE, max_retries=5)
        return float(result.get("score", 0.0))
    except Exception as e:
        print(f"  [Lỗi score_trake_video_match] {e}")
        return 0.0
    finally:
        for p in tmp_paths:
            if os.path.exists(p):
                os.remove(p)


In [ ]:
def run_pipeline(query_text: str, top_k: int = 100, max_per_video: int = 5,
                  rerank_top_n: int = 30, known_task_type: str = None,
                  answer_top_n: int = 100, verbose: bool = True) -> dict:
    """verbose=True (mặc định): in ra output của TỪNG BLOCK trong pipeline (Query
    Understanding, Task Router, 4 module Search, RRF Fusion, Diversify, Temporal
    Reasoning, Rerank, Output cuối) — để quan sát/đánh giá pipeline đang làm gì ở
    từng bước, không cần tách riêng 1 hàm debug khác (từng gây lệch pha với bản
    production thật trước đây). Đặt verbose=False khi chạy batch thật nếu muốn
    log gọn hơn.

    rerank_top_n=30 (P2, tăng từ 20): nếu đáp án đúng nằm ngoài top-20 sau RRF thô (chỉ
    dựa trên 4 module search, chưa qua Gemini xem ảnh thật), nó KHÔNG BAO GIỜ được rerank
    -> mất cơ hội leo hạng dù RRF score ban đầu chỉ hơi thấp. Chi phí escalate hạng 1 lên
    Flash chỉ tốn 1-2 lượt/query nên còn dư địa rate limit để tăng recall theo cách này.
    Cân nhắc thử 40-50 nếu vẫn còn dư thời gian/quota sau khi đo thử với 30."""

    def _log(block_name: str, **kwargs):
        if not verbose:
            return
        print(f"\n{'='*15} {block_name} {'='*15}")
        for k, v in kwargs.items():
            print(f"  {k}: {v}")

    qu = understand_query(query_text)
    if known_task_type: qu["task_type"] = known_task_type

    _log("QUERY UNDERSTANDING (TEXT)",
         task_type=qu.get("task_type"),
         entities=qu.get("entities"),
         actions=qu.get("actions"),
         expanded_queries=qu.get("expanded_queries"),
         question=qu.get("question"),
         needs_temporal=qu.get("needs_temporal"),
         temporal_position_hint=qu.get("temporal_position_hint"),
         N=qu.get("N"))

    # FIX ROUTING: Chỉ bật needs_temporal cho các task thực sự cần chuỗi sự kiện
    is_temporal_task = qu["task_type"] in ["TRAKE", "QA"] or query_text.lower().startswith("e1:")
    if not is_temporal_task:
        qu["needs_temporal"] = False

    needs_temporal = qu.get("needs_temporal", False)
    tools = normalize_tool_selection(qu.get("tool_selection"))

    _log("TASK ROUTER + AGENT PLANNER",
         tool_selection_goc_tu_Gemini=qu.get("tool_selection"),
         tools_sau_chuan_hoa=sorted(tools),
         is_temporal_task=is_temporal_task,
         needs_temporal_sau_routing=needs_temporal)

    v_res = visual_search(qu, top_k=top_k) if "VisualSearch" in tools else pd.DataFrame()
    _log("VISUAL SEARCH", so_candidate=len(v_res))

    t_res = text_search(qu, top_k=top_k) if "TextSearch" in tools else pd.DataFrame()
    _log("TEXT SEARCH", so_candidate=len(t_res),
         tu_OCR=int((t_res["module"] == "text_ocr").sum()) if len(t_res) else 0,
         tu_ASR=int((t_res["module"] == "text_asr").sum()) if len(t_res) else 0)

    o_res = object_search(qu, top_k=top_k) if "ObjectSearch" in tools else pd.DataFrame()
    _log("OBJECT SEARCH", so_candidate=len(o_res))

    a_res = audio_search(qu, top_k=top_k) if "AudioSearch" in tools else pd.DataFrame()
    _log("AUDIO SEARCH", so_candidate=len(a_res))

    # FIX: thêm "audio" vào fusion — trước đây a_res được TÍNH nhưng KHÔNG được đưa vào
    # reciprocal_rank_fusion(), trong khi compute_adaptive_weights() vẫn tính trọng số audio
    # và cộng vào tổng để normalize -> làm loãng trọng số visual/text/object 1 cách vô ích
    # mỗi khi Gemini chọn AudioSearch. reciprocal_rank_fusion() đã tự bỏ qua df rỗng nên an
    # toàn khi audio_search() chưa có model CLAP thật (trả DataFrame rỗng).
    weights = compute_adaptive_weights(qu)
    fused = reciprocal_rank_fusion({"visual": v_res, "text": t_res, "object": o_res, "audio": a_res}, weights=weights)
    _log("RRF FUSION", trong_so=weights, so_candidate_sau_fusion=len(fused))

    diversified = diversify(fused, max_per_video=max_per_video, budget=top_k)
    # apply_temporal_position_filter() (Phần 8.1) — kênh tín hiệu "đầu/giữa/cuối video" DUY
    # NHẤT còn lại cho Textual_KIS sau khi needs_temporal bị ép False phía trên — quan trọng
    # cho các câu kiểu "đoạn clip BẮT ĐẦU VỚI...".
    diversified = apply_temporal_position_filter(diversified, qu.get("temporal_position_hint"))
    _log("DIVERSIFY + POSITION FILTER", so_candidate=len(diversified))

    temporal_result = None
    # Chèn guard kiểm tra task_type để tránh rò rỉ sentinel score vào KIS thông thường
    if needs_temporal and is_temporal_task and len(diversified) > 0:
        top_video_id = diversified.iloc[0]["video_id"]

        if qu["task_type"] == "TRAKE":
            # DANTE (thay thế P1 cũ): TRAKE chấm all-or-nothing THEO VIDEO — sai video mất
            # trắng cả câu. Vì DANTE không tốn Gemini, xét được NHIỀU video ứng viên cùng lúc
            # (không chỉ 2 như P1 cũ) gần như miễn phí, KHÔNG cần điều kiện "hạng 1/2 sát điểm"
            # mới kiểm tra — luôn xét đủ top ứng viên để giảm rủi ro chọn sai video.
            candidate_video_ids = list(dict.fromkeys(diversified["video_id"].tolist()))[:8]
            raw_conditions = qu["scene"]["conditions"]
            normalized_conditions = []
            for i, c in enumerate(raw_conditions):
                if isinstance(c, dict):
                    normalized_conditions.append({
                        "order": c.get("order", i + 1),
                        "action": c.get("action") or c.get("event") or "",
                    })
                else:
                    normalized_conditions.append({"order": i + 1, "action": str(c)})
            events = sorted(normalized_conditions, key=lambda e: e["order"])
            event_texts = [e["action"].replace("_", " ") for e in events]

            _log("DANTE", so_video_ung_vien=len(candidate_video_ids),
                 video_ung_vien=candidate_video_ids, so_event=len(event_texts))
            dante_result = dante_select_best_video(candidate_video_ids, event_texts, verbose=verbose)
            temporal_result = {"video_id": dante_result["video_id"],
                                "frame_ids": dante_result["frame_ids"], "events": events}
            top_video_id = dante_result["video_id"]
        else:
            # QA-temporal: giữ nguyên cơ chế cũ (Gemini-based) — không all-or-nothing theo
            # video như TRAKE nên rủi ro thấp hơn, chưa cần đổi sang DANTE.
            temporal_result = temporal_reasoning(qu, top_video_id)

        _log("TEMPORAL REASONING",
             video_id=top_video_id,
             events=temporal_result.get("events"),
             frame_ids=temporal_result.get("frame_ids"))
        valid_frame_ids = [f for f in temporal_result["frame_ids"] if f is not None]
        if valid_frame_ids:
            temporal_rows = pd.DataFrame([{
                "video_id": top_video_id,
                "frame_id": fid,
                "local_frame_idx": get_local_frame_idx_from_frame_id(top_video_id, fid),
                "rrf_score": 999.0,
                "modules": ["temporal_reasoning"],   # không phải do module search thật tìm ra
            } for fid in valid_frame_ids])
            diversified = pd.concat([temporal_rows, diversified], ignore_index=True).drop_duplicates(["video_id", "frame_id"]).reset_index(drop=True)

    # Lọc bỏ các dòng không có local_frame_idx (không có ảnh để Gemini chấm điểm)
    diversified = diversified[diversified['local_frame_idx'].notna()].copy()

    reranked = multimodal_rerank(diversified, query_text, top_n_to_rerank=rerank_top_n)
    if verbose:
        top5_cols = [c for c in ["video_id", "frame_id", "rerank_score", "rrf_score", "caption"] if c in reranked.columns]
        _log("MULTIMODAL RERANK",
             so_candidate=len(reranked),
             top_5=reranked[top5_cols].head(5).to_dict("records") if len(reranked) else [])

    # FIX (phát hiện qua test thật — câu múa lân): DANTE chọn video CHỈ dựa vào điểm CLIP thô
    # (S[i][t] + DP) — 2 video ứng viên có thể chênh nhau rất ít (VD 1.3465 vs 1.3369, ~1%) dù
    # về NỘI DUNG THẬT khác hẳn nhau, vì CLIP không đủ sắc bén để phân biệt các video cùng chủ
    # đề (nhiều đội múa lân, cùng bối cảnh: lân trắng, cột trụ, cờ trắng viền đỏ). Gemini Vision
    # (rerank_one(), mạnh hơn hẳn CLIP) đã CHẤM SẴN mọi candidate trong reranked — bao gồm cả
    # frame DANTE tự chọn (đánh dấu rrf_score=999.0) LẪN các frame từ video KHÁC (do Visual/
    # Object Search độc lập tìm ra). Nếu 1 video KHÁC được Gemini chấm cao vượt trội so với
    # chính video DANTE chọn, đây là bằng chứng mạnh cho thấy DANTE chọn sai video — chạy lại
    # DANTE CHỈ cho đúng video đó (rẻ, không tốn thêm Gemini) để lấy đúng chuỗi frame.
    if qu["task_type"] == "TRAKE" and temporal_result is not None and len(reranked) > 0:
        dante_rows = reranked[reranked["rrf_score"] == 999.0]
        dante_video_best_score = float(dante_rows["rerank_score"].max()) if len(dante_rows) else 0.0

        other_rows = reranked[reranked["video_id"] != temporal_result["video_id"]]
        if len(other_rows) > 0:
            other_best = other_rows.loc[other_rows["rerank_score"].idxmax()]
            if other_best["rerank_score"] >= 0.7 and other_best["rerank_score"] > dante_video_best_score + 0.3:
                _log("DANTE — PHÁT HIỆN SAI VIDEO",
                     video_dante_chon=temporal_result["video_id"], diem_gemini_cho_video_do=dante_video_best_score,
                     video_reranker_de_xuat=other_best["video_id"], diem_gemini_video_do=float(other_best["rerank_score"]))
                event_texts_correction = [e["action"] for e in temporal_result["events"]]
                corrected = dante_align_events(other_best["video_id"], event_texts_correction, verbose=verbose)
                temporal_result = {"video_id": corrected["video_id"], "frame_ids": corrected["frame_ids"],
                                    "events": temporal_result["events"]}
                _log("DANTE — SAU KHI SỬA", video_id=temporal_result["video_id"], frame_ids=temporal_result["frame_ids"])

    output = output_formatter(qu, reranked, temporal_result=temporal_result, answer_top_n=answer_top_n)
    verified = final_verification(output)

    if verbose:
        n_results = len(verified.get("ranked_results", verified.get("frame_ids", [])))
        _log("OUTPUT CUỐI", task_type=verified.get("task_type"), so_dong_ket_qua=n_results)

    return verified


---
## Phần 15 — Result/Submission Layer: xuất file ĐÚNG CHUẨN nộp BTC

**Giải thích:** đối chiếu trực tiếp với quy định nộp bài thật tại [sotuyenaic.oj.io.vn](https://sotuyenaic.oj.io.vn/), đã sửa lại toàn bộ cho khớp:
- **KHÔNG có header row**, **KHÔNG có cột `rank`** — chỉ đúng số cột theo từng loại.
- **QA**: answer cắt tối đa **100 ký tự**.
- **Tối đa 100 dòng/file**.
- **Tên file PHẢI khớp đúng tên BTC cấp** (VD `query-1-kis.txt` → `query-1-kis.csv`) — không tự đặt tên tùy ý.
- **Đóng gói đúng cấu trúc**: file `.zip` phải chứa 1 thư mục con tên `submission/`, bên trong là các CSV — không nén trực tiếp CSV.
- **Đã bỏ hẳn** phần "mô tả phương pháp" (`.txt` kèm theo) — quy định thật KHÔNG yêu cầu điều này (giả định trước đó không đúng thực tế).

In [99]:
import zipfile


def generate_trake_jitter_rows(video_id: str, frame_ids: list, max_rows: int = 100,
                                offsets: list = None) -> list:
    """CẢI TIẾN (P1.1): TRAKE chấm all-or-nothing theo video (sai video = 0 điểm toàn bộ) và
    khoảng đáp án đúng mỗi event RẤT HẸP (< 10 frame — xem quy chế chấm điểm). Trước đây
    export_submission() chỉ ghi ĐÚNG 1 dòng cho TRAKE dù BTC cho phép tối đa 100 dòng/file —
    bỏ phí hoàn toàn cơ chế R@5/R@20/R@50/R@100 (R@k = điểm cao nhất trong k dòng đầu, không
    giảm theo k -> càng nhiều phương án hợp lý càng có lợi, không mất gì).

    Hàm này sinh thêm các dòng BIẾN THỂ bằng cách dịch TOÀN BỘ N frame_id theo CÙNG 1 offset nhỏ
    (không tốn thêm lần gọi Gemini/ffmpeg nào — chỉ cộng trừ số nguyên) để hedge cho sai số
    alignment nhỏ mang tính hệ thống. Dòng đầu tiên LUÔN là bộ frame_id gốc (offset=0) vì đây là
    phương án tự tin nhất, cần ưu tiên cho R@1.

    LƯU Ý: cách này KHÔNG hedge được cho trường hợp đoán SAI VIDEO (rủi ro lớn nhất vì mất trắng
    điểm) — muốn hedge việc đó cần chạy thêm temporal_reasoning() cho video ứng viên hạng 2/3,
    tốn thêm API call đáng kể nên chưa tự động hoá ở đây, cân nhắc bổ sung riêng nếu quota cho
    phép."""
    if offsets is None:
        offsets = [0, -3, 3, -6, 6, -9, 9, -12, 12, -15, 15, -20, 20, -25, 25, -30, 30]

    rows, seen = [], set()
    for offset in offsets:
        if len(rows) >= max_rows:
            break
        shifted = []
        for fid in frame_ids:
            if fid is None:
                shifted.append(None)
            else:
                shifted.append(max(0, int(fid) + offset))
        key = tuple(shifted)
        if key in seen:
            continue   # bỏ qua nếu clip về 0 khiến trùng với dòng đã có
        seen.add(key)
        rows.append([video_id] + shifted)
    return rows


def export_submission(result: dict, output_path: str) -> pd.DataFrame:
    """Xuất kết quả thành file CSV ĐÚNG ĐỊNH DẠNG BTC yêu cầu (xác nhận qua
    https://sotuyenaic.oj.io.vn/):
    - KHÔNG header, KHÔNG cột 'rank' (thứ tự dòng = thứ hạng, không cần ghi số)
    - Textual KIS : <video_id>,<frame_id>
    - QA          : <video_id>,<frame_id>,<answer>   (answer tối đa 100 ký tự)
    - TRAKE       : <video_id>,<frame_id_1>,...,<frame_id_N>
    - Tối đa 100 dòng, encoding UTF-8
    """
    task_type = result["task_type"]

    if task_type == "TRAKE":
        if result.get("video_id") is None:
            df = pd.DataFrame()   # không xác định được video -> nộp rỗng, không đoán bừa
        else:
            rows = generate_trake_jitter_rows(result["video_id"], result["frame_ids"], max_rows=100)
            df = pd.DataFrame(rows)
    else:
        rows = []
        for r in result["ranked_results"][:100]:   # BTC: tối đa 100 dòng
            if task_type == "QA":
                answer = (r.get("answer") or "")[:100]   # BTC: answer tối đa 100 ký tự
                rows.append([r["video_id"], r["frame_id"], answer])
            else:   # Textual_KIS
                rows.append([r["video_id"], r["frame_id"]])
        df = pd.DataFrame(rows)

    df.to_csv(output_path, index=False, header=False, encoding="utf-8")
    print(f"Đã xuất file nộp bài: {output_path} ({len(df)} dòng)")
    return df


def package_submission_zip(csv_dir: str, zip_output_path: str):
    """Đóng gói TOÀN BỘ file CSV trong csv_dir thành 1 file .zip ĐÚNG cấu trúc
    BTC yêu cầu: file zip PHẢI chứa 1 thư mục con tên 'submission/' bên trong,
    KHÔNG được nén trực tiếp các file CSV (lỗi thường gặp #2 theo trang hướng dẫn)."""
    csv_files = sorted(Path(csv_dir).glob("*.csv"))
    if not csv_files:
        print(f"[CẢNH BÁO] Không tìm thấy file CSV nào trong {csv_dir}")
        return

    with zipfile.ZipFile(zip_output_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for csv_path in csv_files:
            zf.write(csv_path, arcname=f"submission/{csv_path.name}")   # ĐÚNG cấu trúc thư mục con

    print(f"Đã đóng gói {len(csv_files)} file CSV vào: {zip_output_path}")
    print(f"Cấu trúc bên trong zip: submission/{[p.name for p in csv_files]}")

---
### 15.0b — Hiển thị kết quả đúng format BTC + đóng gói ảnh top N tải về

**Giải thích:** sau khi `export_submission()` ghi CSV, hàm này (1) in ra ĐÚNG nội dung sẽ nằm trong file CSV (không header, đúng thứ tự dòng — y hệt BTC sẽ nhận), để bạn xem trực tiếp trong notebook không cần mở file, và (2) copy ảnh keyframe thật (KIS/QA) hoặc trích frame trực tiếp từ video (TRAKE, vì frame TRAKE thường không phải keyframe có sẵn) cho tối đa 100 dòng đầu, đóng gói cùng CSV thành 1 file `.zip` — tải về được ngay để xem lại ảnh bằng mắt mà không cần mở lại Drive.


In [ ]:
def assess_confidence(rerank_score, modules) -> str:
    """Đánh giá độ tin cậy 1 candidate dựa trên 2 tín hiệu MIỄN PHÍ (không tốn thêm API call),
    độc lập với retrieval:
    1. rerank_score — Gemini Vision tự chấm khớp ảnh-query lại từ đầu (0-1).
    2. Đồng thuận module — bao nhiêu module ĐỘC LẬP (visual/text/object/audio) cùng tìm ra
       candidate này (KHÔNG tính "neighbor_expansion"/"temporal_reasoning" — đây là candidate
       suy ra/tổng hợp, không phải do module search thật sự tìm ra).
    CAO: rerank_score >= 0.7 VÀ có từ 2 module thật đồng thuận trở lên.
    THẤP: rerank_score < 0.4 VÀ chỉ có 0-1 module thật.
    TRUNG BÌNH: các trường hợp còn lại."""
    real_modules = [m for m in (modules or []) if m in ("visual", "text", "object", "audio")]
    n_modules = len(real_modules)
    score = rerank_score if rerank_score is not None and pd.notna(rerank_score) else 0.0

    if score >= 0.7 and n_modules >= 2:
        return "CAO"
    if score < 0.4 and n_modules <= 1:
        return "THẤP"
    return "TRUNG BÌNH"


def display_and_package_result(stem: str, query_text: str, result: dict,
                                submission_df: pd.DataFrame, output_folder: str,
                                max_images: int = 100, auto_download: bool = True,
                                n_confidence_report: int = 5) -> str:
    """Hiển thị kết quả đúng format BTC (không header, đúng thứ tự dòng — y hệt CSV sẽ nộp)
    + báo cáo độ tin cậy cho top n_confidence_report candidate (rerank_score + caption + đồng
    thuận module — xem assess_confidence()) + đóng gói ảnh top N kèm CSV thành 1 file .zip.
    auto_download=True: tự bật popup tải về ngay (Colab) — đặt False khi chạy batch nhiều câu
    để tránh spam nhiều popup tải cùng lúc, file zip vẫn được tạo ra bình thường, tải thủ công
    sau từ output_folder.
    """
    print(f"\n{'='*15} KẾT QUẢ CUỐI CÙNG — {stem} (đúng format BTC) {'='*15}")
    print(f"Câu hỏi: {query_text}\n")
    if len(submission_df) == 0:
        print("(Không có dòng nào — xem log các block phía trên để biết lý do)")
    else:
        for _, row in submission_df.iterrows():
            print(",".join("" if pd.isna(v) else str(v) for v in row.tolist()))

    # ---- Báo cáo độ tin cậy (miễn phí — dùng lại rerank_score/caption/modules đã tính sẵn) ----
    ranked_results = result.get("ranked_results")
    if ranked_results:
        print(f"\n{'='*15} ĐỘ TIN CẬY (top {min(n_confidence_report, len(ranked_results))}) {'='*15}")
        for r in ranked_results[:n_confidence_report]:
            level = assess_confidence(r.get("rerank_score"), r.get("modules"))
            real_modules = [m for m in (r.get("modules") or []) if m in ("visual", "text", "object", "audio")]
            print(f"  --- hạng {r.get('rank')} — {r.get('video_id')} frame {r.get('frame_id')} ---")
            print(f"    rerank_score: {r.get('rerank_score')}")
            print(f"    caption: {r.get('caption')}")
            print(f"    đồng thuận module: {', '.join(real_modules) if real_modules else '(không có / suy ra từ candidate khác)'}"
                  f"  ({len(real_modules)} module độc lập)")
            print(f"    => Mức tin cậy: {level}")
    elif result.get("task_type") == "TRAKE":
        print(f"\n{'='*15} ĐỘ TIN CẬY {'='*15}")
        print("  TRAKE không có rerank_score/modules theo từng frame (frame đến từ Temporal")
        print("  Reasoning + Dense Frame Access, không qua multimodal_rerank()) — tự xem ảnh")
        print("  trong file zip tải về bên dưới để đánh giá.")

    # ---- Chuẩn bị thư mục ảnh (dọn sạch nếu chạy lại) ----
    images_dir = os.path.join(output_folder, f"{stem}_images")
    os.makedirs(images_dir, exist_ok=True)
    for f in Path(images_dir).glob("*"):
        f.unlink()

    task_type = result["task_type"]
    n_saved = 0

    if task_type == "TRAKE":
        # Frame TRAKE thường KHÔNG phải keyframe có sẵn (đến từ Dense Frame Access) ->
        # phải trích trực tiếp từ video bằng extract_single_frame(), không dùng
        # get_keyframe_image_path() như nhánh KIS/QA bên dưới.
        video_id = result.get("video_id")
        frame_ids = result.get("frame_ids", [])
        if video_id is not None:
            video_path = find_video_file(video_id)
            if video_path:
                for i, fid in enumerate(frame_ids[:max_images]):
                    if fid is None:
                        continue
                    out_path = os.path.join(images_dir, f"event_{i+1:02d}_{video_id}_frame{fid}.jpg")
                    extract_single_frame(video_path, fid, out_path)
                    if os.path.exists(out_path):
                        n_saved += 1
    else:
        for rank, r in enumerate(result.get("ranked_results", [])[:max_images], start=1):
            local_idx = r.get("local_frame_idx")
            if local_idx is None or pd.isna(local_idx):
                continue
            src_path = get_keyframe_image_path(r["video_id"], local_idx)
            if src_path is None or not os.path.exists(src_path):
                continue
            ext = os.path.splitext(src_path)[1] or ".jpg"
            out_path = os.path.join(images_dir, f"rank_{rank:03d}_{r['video_id']}_frame{r['frame_id']}{ext}")
            shutil.copy(src_path, out_path)
            n_saved += 1

    print(f"\nĐã lưu {n_saved} ảnh vào: {images_dir}")

    # ---- Đóng gói CSV + ảnh thành 1 zip ----
    zip_path = os.path.join(output_folder, f"{stem}_result_package.zip")
    csv_path = os.path.join(output_folder, f"{stem}.csv")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        if os.path.exists(csv_path):
            zf.write(csv_path, arcname=f"{stem}.csv")
        for img_path in sorted(Path(images_dir).glob("*")):
            zf.write(img_path, arcname=f"images/{img_path.name}")

    print(f"Đã đóng gói: {zip_path} (CSV + {n_saved} ảnh)")

    if auto_download:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception as e:
            print(f"[LƯU Ý] Không tự tải được ({e}) — file vẫn nằm tại: {zip_path}, "
                  f"tải thủ công từ khung Files bên trái Colab.")

    return zip_path


### 15.1 (MỚI) — Quy trình THẬT: xử lý 1 gói câu hỏi BTC cấp

**Cách dùng lúc thi thật:**
1. Tải các file `.txt` BTC cấp (VD `query-1-kis.txt`, `query-2-qa.txt`, `query-3-trake.txt`) vào 1 thư mục, VD `/content/query_package/`.
2. Chạy `process_query_package()` — tự đọc từng file, tự nhận diện loại qua HẬU TỐ tên file (`kis`/`qa`/`trake`), chạy pipeline, xuất CSV **ĐÚNG TÊN** (chỉ đổi đuôi `.txt` → `.csv`).
3. Chạy `package_submission_zip()` — đóng gói đúng cấu trúc `submission/` rồi zip.
4. Tải file `.zip` về, đăng nhập hệ thống BTC, nộp trực tiếp.

In [ ]:
SUFFIX_TO_TASK_TYPE = {"kis": "Textual_KIS", "qa": "QA", "trake": "TRAKE"}


def process_query_package(query_folder: str, output_folder: str,
                           top_k: int = 100, rerank_top_n: int = 30,   # P2: tăng từ 20 (xem giải thích ở run_pipeline())
                           auto_download_each: bool = False) -> dict:
    """Xử lý TOÀN BỘ 1 gói câu hỏi BTC cấp — đọc từng file .txt, tự nhận diện
    task_type qua HẬU TỐ tên file (đúng quy ước BTC: '...-kis.txt' / '...-qa.txt' /
    '...-trake.txt'), chạy pipeline với known_task_type ÉP BUỘC (không để LLM tự
    đoán sai), xuất CSV ĐÚNG TÊN tương ứng.

    auto_download_each=False (mặc định): KHÔNG tự bật popup tải zip cho từng câu khi chạy
    batch nhiều câu (tránh spam nhiều popup cùng lúc) — file zip (CSV + ảnh top N) vẫn được
    tạo ra bình thường cho MỖI câu trong output_folder, tải thủ công sau. Đặt True nếu chỉ
    chạy 1-2 câu để test và muốn tải ngay.

    TRẢ VỀ: dict {stem: {"query_text":..., "result":...}} — dùng để hiển thị ảnh
    kiểm tra kết quả THẬT ngay sau đó (Phần 14), không cần câu hỏi mẫu tự đặt."""
    os.makedirs(output_folder, exist_ok=True)
    query_files = sorted(Path(query_folder).glob("*.txt"))
    print(f"Tìm thấy {len(query_files)} file câu hỏi trong {query_folder}\n")

    all_results = {}
    for qf in query_files:
        stem = qf.stem   # VD "query-1-kis"
        suffix = stem.rsplit("-", 1)[-1].lower()   # lấy đúng hậu tố cuối: "kis"/"qa"/"trake"
        known_type = SUFFIX_TO_TASK_TYPE.get(suffix)
        if known_type is None:
            print(f"[CẢNH BÁO] Không nhận diện được loại truy vấn từ tên file '{qf.name}' "
                  f"(hậu tố '{suffix}' không khớp kis/qa/trake) -> BỎ QUA file này")
            continue

        query_text = qf.read_text(encoding="utf-8").strip()
        print(f"{'='*15} {stem} (task_type={known_type}) {'='*15}")
        print(f"Câu hỏi: {query_text}")

        output_csv = os.path.join(output_folder, f"{stem}.csv")   # GIỮ ĐÚNG TÊN GỐC

        # SỬA LỖI (P0.2 — BUG NGHIÊM TRỌNG): TRƯỚC ĐÂY vòng lặp này KHÔNG có try/except -> 1
        # câu hỏi bị lỗi (VD NameError, timeout API, video thiếu file...) sẽ dừng NGANG vòng
        # lặp, khiến TẤT CẢ câu hỏi PHÍA SAU trong cùng gói KHÔNG được xử lý/xuất CSV luôn (mất
        # trắng nhiều câu, không phải chỉ câu lỗi). Giờ: lỗi ở 1 câu chỉ làm câu đó nhận CSV
        # rỗng hợp lệ format (0 điểm), các câu khác vẫn chạy bình thường.
        try:
            # answer_top_n=20: giới hạn số candidate QA thực sự được hỏi Gemini — khớp
            # đúng mốc chấm điểm R@20 (mặc định cũ answer_top_n=100 tốn gấp 5 lần API
            # call cho lợi ích cận biên rất nhỏ ở R@50/R@100), giúp giảm đáng kể rủi ro
            # chạm rate limit 15 request/phút của free tier.
            result = run_pipeline(query_text, top_k=top_k, rerank_top_n=rerank_top_n,
                                   known_task_type=known_type, answer_top_n=20)
            submission_df = export_submission(result, output_csv)
            display_and_package_result(stem, query_text, result, submission_df, output_folder,
                                        auto_download=auto_download_each)
            all_results[stem] = {"query_text": query_text, "result": result}
        except Exception as e:
            import traceback
            print(f"  [LỖI NGHIÊM TRỌNG] Xử lý câu '{stem}' thất bại: {e}")
            traceback.print_exc()
            pd.DataFrame().to_csv(output_csv, index=False, header=False, encoding="utf-8")
            print(f"  [Fallback] Đã ghi CSV RỖNG (đúng format, 0 dòng) cho '{stem}' để KHÔNG "
                  f"thiếu file trong gói nộp — câu này sẽ được 0 điểm nhưng KHÔNG ảnh hưởng "
                  f"các câu khác trong gói.")
            fallback_result = ({"task_type": known_type, "video_id": None, "frame_ids": []}
                                if known_type == "TRAKE"
                                else {"task_type": known_type, "ranked_results": []})
            all_results[stem] = {"query_text": query_text, "result": fallback_result}

        print()

    return all_results

### 15.2 — Chạy pipeline để lấy kết quả nộp bài

In [101]:
import os
from google.colab import files

# 1. Định nghĩa lại các đường dẫn lưu trữ tại /content
QUERY_PACKAGE_DIR = "/content/query_package"
SUBMISSION_CSV_DIR = "/content/submission_results"
SUBMISSION_ZIP_PATH = "/content/team_submission.zip"

# Đảm bảo các thư mục tồn tại
os.makedirs(QUERY_PACKAGE_DIR, exist_ok=True)
os.makedirs(SUBMISSION_CSV_DIR, exist_ok=True)

package_results = {}

# 2. Xử lý pipeline
if any(Path(QUERY_PACKAGE_DIR).glob("*.txt")):
    print(f"--- Bắt đầu xử lý các file trong {QUERY_PACKAGE_DIR} ---")
    package_results = process_query_package(QUERY_PACKAGE_DIR, SUBMISSION_CSV_DIR)

    # 3. Đóng gói và tự động tải về
    package_submission_zip(SUBMISSION_CSV_DIR, SUBMISSION_ZIP_PATH)

    if os.path.exists(SUBMISSION_ZIP_PATH):
        print(f"\n--- Hoàn tất! Đang tự động tải file: {SUBMISSION_ZIP_PATH} ---")
        files.download(SUBMISSION_ZIP_PATH)
else:
    print(f"Thư mục {QUERY_PACKAGE_DIR} đang trống.")
    print("HÃY UPLOAD CÁC FILE .txt CỦA BTC VÀO THƯ MỤC NÀY RỒI CHẠY LẠI CELL.")

Thư mục /content/query_package đang trống.
HÃY UPLOAD CÁC FILE .txt CỦA BTC VÀO THƯ MỤC NÀY RỒI CHẠY LẠI CELL.


In [102]:
from google.colab import files
import os

# Sử dụng đường dẫn đã định nghĩa ở các cell trước
if os.path.exists(SUBMISSION_ZIP_PATH):
    print(f"Đang tải xuống: {SUBMISSION_ZIP_PATH}")
    files.download(SUBMISSION_ZIP_PATH)
else:
    print(f"LỖI: Không tìm thấy file tại {SUBMISSION_ZIP_PATH}. Vui lòng chạy cell Phần 15.2 trước để tạo file zip.")

LỖI: Không tìm thấy file tại /content/team_submission.zip. Vui lòng chạy cell Phần 15.2 trước để tạo file zip.
